## Mount your Google Drive and install necessary libraries

Requirements:
1. [Google Account](https://www.google.com/account/about/)
2. [Google Drive](https://www.google.com/drive/)

In [2]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -U -q segmentation-models
!pip install -q tensorflow
!pip install -q keras
!pip install -q -U keras-tuner
!pip install -q rasterio geopandas contextily
!pip install opencv-python-headless
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
 # This line will install the packages/libraries which are not present in Google Colab. Remember that once the session runtime is over, you need to run this cell as well as the one above again.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 100.4 MB/s eta 0:00:00


In [4]:
import tensorflow as tf
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import cv2
import rasterio
import geopandas as gpd
import contextily as cx
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import metrics
from rasterio.plot import show
from tensorflow.keras.layers import Input, Conv2D, GroupNormalization, MaxPooling2D, MaxPool2D, UpSampling2D, concatenate, Conv2DTranspose, BatchNormalization, Dropout, Softmax, Multiply,DepthwiseConv2D, Layer
from tensorflow.keras.models import Model
from keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, multiply, Permute, Concatenate, Conv2D, Add, Activation, Lambda, LayerNormalization
from tensorflow.keras.metrics import Precision, Recall, F1Score, MeanIoU
from keras import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K
from tensorflow.keras import backend as K
from typing import Callable
from sklearn.model_selection import KFold
from tensorflow.keras.utils import plot_model
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, cohen_kappa_score
import time
import gc
from keras.activations import sigmoid, relu
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.optimizers import Adam
from keras import initializers
from typing import Callable
# Library with segmentation metrics
import segmentation_models as sm

import sys
sys.path.append('/content/drive/MyDrive/landslide')

physical_devices = tf.config.experimental.list_physical_devices('GPU')

for device in physical_devices:
    tf.config.experimental.set_memory_growth(device, True)

if tf.test.gpu_device_name():

    print('Default GPU Device:{}'.format(tf.test.gpu_device_name()))
else:
   print("Please install GPU version of TF")

Segmentation Models: using `tf.keras` framework.
Default GPU Device:/device:GPU:0


In [4]:
X_test1 = np.load(f"/content/drive/MyDrive/landslide/Data/Arrays/X_pre_inf.npy")

In [5]:
X_test1 = np.load(f"/content/drive/MyDrive/landslide/Data/Arrays/X_genpre_inf.npy")

In [47]:
X_test1 = np.load(f"/content/drive/MyDrive/landslide/Data/Arrays/X_genpre_inf1.npy")

In [6]:
X_test2 = np.load(f"/content/drive/MyDrive/landslide/Data/Arrays/X_post_inf.npy")
X_test3 = np.load(f"/content/drive/MyDrive/landslide/Data/Arrays/X_topo_inf.npy")
y_test = np.load(f"/content/drive/MyDrive/landslide/Data/Arrays/Y_mask_inf.npy")
labels = np.load(f"/content/drive/MyDrive/landslide/Data/Arrays/label_.npy")

In [7]:
y_test = y_test.astype(np.float32)

In [8]:
print(X_test1.min())
print(X_test1.max())
print(X_test2.min())
print(X_test2.max())
print(X_test3.min())
print(X_test3.max())
print(labels)

0.0
1.0
0.0
1.0
0.0
1.0
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 1 2 3 3 3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]


In [9]:
# Get unique ROI values
unique_rois = np.unique(labels)

# Create a dictionary to store images for each ROI
roi_images = {roi: [] for roi in unique_rois}

# Populate the dictionary with image indices for each ROI
for i, roi in enumerate(labels):
    roi_images[roi].append(i)

# Iterate through each ROI and display a random image
for roi in unique_rois:
    # Select a random image index from the current ROI
    random_image_index = random.choice(roi_images[roi])
    img = random_image_index

    # Create subplots for post-event and pre-event images
    fig, (ax1, ax2) = plt.subplots(2, 10, figsize=(30, 10))

    # --- Post-event images (ax1) ---
    ax1[0].set_title(f"Post RGB (ROI {roi})", fontsize=15)
    ax1[1].set_title(f"Post NIR (ROI {roi})", fontsize=15)
    ax1[2].set_title(f"Post SWIR1 (ROI {roi})", fontsize=15)
    ax1[3].set_title(f"Post SWIR2 (ROI {roi})", fontsize=15)
    ax1[4].set_title(f"Post NDVI (ROI {roi})", fontsize=15)
    ax1[5].set_title(f"Post NDWI (ROI {roi})", fontsize=15)
    ax1[6].set_title(f"Post BI (ROI {roi})", fontsize=15)
    ax1[7].set_title(f"Post BSI (ROI {roi})", fontsize=15)
    ax1[8].set_title(f"Post SAVI (ROI {roi})", fontsize=15)
    ax1[9].set_title(f"Mask (ROI {roi})", fontsize=15)
    ax1[0].imshow(X_test2[img, :, :, :3])
    ax1[1].imshow(X_test2[img, :, :, 3])
    ax1[2].imshow(X_test2[img, :, :, 4])
    ax1[3].imshow(X_test2[img, :, :, 5])
    ax1[4].imshow(X_test2[img, :, :, 6])
    ax1[5].imshow(X_test2[img, :, :, 7])
    ax1[6].imshow(X_test2[img, :, :, 8])
    ax1[7].imshow(X_test2[img, :, :, 9])
    ax1[8].imshow(X_test2[img, :, :, 10])
    ax1[9].imshow(y_test[img, :, :, 0])
    for j in range(10):
        ax1[j].set(xticks=[], yticks=[])

    ax2[0].set_title(f"Pre RGB (ROI {roi})", fontsize=15)
    ax2[1].set_title(f"Pre NIR (ROI {roi})", fontsize=15)
    ax2[2].set_title(f"Pre SWIR1 (ROI {roi})", fontsize=15)
    ax2[3].set_title(f"Pre SWIR2 (ROI {roi})", fontsize=15)
    ax2[4].set_title(f"Pre NDVI (ROI {roi})", fontsize=15)
    ax2[5].set_title(f"Pre NDWI (ROI {roi})", fontsize=15)
    ax2[6].set_title(f"Pre BI (ROI {roi})", fontsize=15)
    ax2[7].set_title(f"Pre BSI (ROI {roi})", fontsize=15)
    ax2[8].set_title(f"Pre SAVI (ROI {roi})", fontsize=15)
    ax2[9].set_title(f"DSM+Slope (ROI {roi})", fontsize=15) # Assuming X_test3 is DSM+Slope
    ax2[0].imshow(X_test1[img, :, :, :3])
    ax2[1].imshow(X_test1[img, :, :, 3])
    ax2[2].imshow(X_test1[img, :, :, 4])
    ax2[3].imshow(X_test1[img, :, :, 5])
    ax2[4].imshow(X_test1[img, :, :, 6])
    ax2[5].imshow(X_test1[img, :, :, 7])
    ax2[6].imshow(X_test1[img, :, :, 8])
    ax2[7].imshow(X_test1[img, :, :, 9])
    ax2[8].imshow(X_test1[img, :, :, 10])
    ax2[9].imshow(X_test3[img, :, :, :3]) # Displaying first 3 channels of X_test3

    # Remove ticks for all subplots
    for j in range(10):
        ax2[j].set(xticks=[], yticks=[])

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

# **INITIALIZE MODEL TRAINING PARAMETERS**

#### **Precision** is a measure of how many of the positive predictions made are correct (true positives).

#### **Recall** is a measure of how many of the positive cases the model correctly predicted, over all the positive cases in the data. It is sometimes also referred to as Sensitivity.

#### **F1-Score** is a measure combining both precision and recall. It is generally described as the harmonic mean of the two. Harmonic mean is just another way to calculate an “average” of values, generally described as more suitable for ratios (such as precision and recall) than the traditional arithmetic mean. The formula used for F1-score in this case is:

In [10]:
class CohenKappaMetric(tf.keras.metrics.Metric):
    def __init__(self, num_classes, name="cohen_kappa", **kwargs):
        super(CohenKappaMetric, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.confusion_matrix = self.add_weight(
            name="confusion_matrix", shape=(num_classes, num_classes), initializer="zeros"
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        # Flatten the predictions and labels to treat each pixel as an independent class prediction
        y_true = tf.reshape(y_true, [-1])
        y_pred = tf.argmax(tf.reshape(y_pred, [-1, self.num_classes]), axis=-1)

        # Update the confusion matrix with the new batch's data
        new_confusion_matrix = tf.math.confusion_matrix(y_true, y_pred, num_classes=self.num_classes)
        self.confusion_matrix.assign_add(new_confusion_matrix)

    def result(self):
        cm = self.confusion_matrix
        n = tf.reduce_sum(cm)
        sum_po = tf.reduce_sum(tf.linalg.diag_part(cm)) / n
        sum_pe = tf.reduce_sum(tf.reduce_sum(cm, axis=0) * tf.reduce_sum(cm, axis=1)) / (n * n)
        kappa = (sum_po - sum_pe) / (1 - sum_pe)
        return kappa

    def reset_states(self):
        for v in self.variables:
            v.assign(tf.zeros_like(v))

In [11]:
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']

#**WHAT IS A LOSS FUNCTION??**

A loss function is a function that compares the target and predicted output values; measures how well the neural network models the training data. When training, we aim to minimize this loss between the predicted and target outputs.

Dice Loss is widely used in image segmentation tasks to address the data imbalance problem.

![image](http://mriquestions.com/uploads/3/4/5/7/34572113/steepest-descent-loss-funciton_orig.png)

In [12]:
available_losses = [
    'categorical_crossentropy',
    'classical_jaccard',
    'binary_crossentropy',
    'jaccard_power_20', # p = 2
    'classical_dice',
    'dice_power_15', # p = 1.5
    'dice_power_20', # p = 2
    'dice_loss',
    'balanced_lovasz_tversky_loss'
]

In [13]:
selected_loss = available_losses[8]
print("selected loss:", selected_loss)

selected loss: balanced_lovasz_tversky_loss


In [14]:
@tf.keras.utils.register_keras_serializable()
def jaccard_pow_loss(y_true, y_pred, p_value=2.0,smooth = 1):
    p_value = POWER_VALUE
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)

    intersection = K.sum(y_true_f * y_pred_f)
    term_true = K.sum(K.pow(y_true_f, p_value))
    term_pred = K.sum(K.pow(y_pred_f, p_value))
    union = term_true + term_pred - intersection
    return 1 - ((intersection + smooth) / (union + smooth))

def dice_pow_loss(y_true, y_pred,p_value=2.0, smooth = 1):
    p_value = POWER_VALUE
    y_true=K.flatten(y_true)
    y_pred=K.flatten(y_pred)
    numerator=K.sum(2*(y_true * y_pred))
    y_true = K.pow(y_true, p_value)
    y_pred = K.pow(y_pred, p_value)
    denominator = K.sum(y_true) +  K.sum(y_pred)
    return (1-((numerator+smooth)/(denominator+smooth)))


def dsc(y_true, y_pred, smooth = 1):
    smooth = 1.
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    score = (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
    return score

def dice_loss(y_true, y_pred):
    loss = 1 - dsc(y_true, y_pred)
    return loss

@tf.keras.utils.register_keras_serializable()
def balanced_lovasz_tversky_loss(y_true, y_pred, alpha=0.3, beta=0.7):

    def lovasz_sigmoid(y_pred, y_true):
        # Compute the Lovász-Sigmoid loss
        errors = y_true - y_pred
        errors_sorted, perm = tf.nn.top_k(errors, k=tf.shape(errors)[0])  # tf.nn.top_k likely needs tf
        y_true_sorted = tf.gather(y_true, perm) # tf.gather likely needs tf
        grad = K.cumsum(y_true_sorted, axis=0)  # Use K.cumsum
        loss = K.sum(K.relu(errors_sorted - grad)) # Use K.sum and K.relu
        return loss

    def tversky_loss(y_pred, y_true, alpha, beta):
        # Calculate the Tversky index
        numerator = K.sum(y_pred * y_true) # Use K.sum
        denominator = numerator + alpha * K.sum(y_pred * (1 - y_true)) + beta * K.sum((1 - y_pred) * y_true) # Use K.sum

        # Tversky loss
        tversky_loss = 1 - (numerator / denominator)
        return tversky_loss

    # Flatten tensors for easier computation
    y_true = K.flatten(y_true) # Use K.flatten
    y_pred = K.flatten(y_pred) # Use K.flatten

    # Calculate the losses
    lovasz_loss = lovasz_sigmoid(y_pred, y_true)
    tversky_loss = tversky_loss(y_pred, y_true, alpha, beta)

    # Combine the losses
    combined_loss = lovasz_loss + tversky_loss
    return combined_loss

In [15]:
if selected_loss == 'categorical_crossentropy':
    loss = selected_loss
    POWER_VALUE = 0.0
elif selected_loss == 'classical_jaccard':
    loss = jaccard_pow_loss
    POWER_VALUE = 1.0
elif selected_loss == 'binary_crossentropy':
    loss = selected_loss
elif selected_loss == 'jaccard_power_20':
    loss = jaccard_pow_loss
    POWER_VALUE = 2.0
elif selected_loss == 'classical_dice':
    loss = dice_pow_loss
    POWER_VALUE = 1.0
elif selected_loss == 'dice_power_15':
    loss = dice_pow_loss
    POWER_VALUE = 1.5
elif selected_loss == 'dice_power_20':
    loss = dice_pow_loss
    POWER_VALUE = 2.0
elif selected_loss == 'dice_power_20':
    loss = dice_loss
elif selected_loss == 'dice_loss':
    loss = dice_loss
elif selected_loss == 'balanced_lovasz_tversky_loss':
    loss = balanced_lovasz_tversky_loss

In [16]:
print("selected_loss", loss)
#print("p_value", POWER_VALUE)

selected_loss <function balanced_lovasz_tversky_loss at 0x7c13b0691b20>


## DEFINITION CLASS API FOR THE MODEL


*   Global Response Normalization (GRN)+Droppath

*   SequentialPolarizedSelfAttention (SPA)
*   SimplePooling (Avg-K pooling)


*   Attention (Spatial and Channel)

*   Resize









In [17]:
class SpectralNorm(tf.keras.constraints.Constraint):
    def __init__(self, n_iter=5):
        self.n_iter = n_iter

    def call(self, input_weights):
        w = tf.reshape(input_weights, (-1, input_weights.shape[-1]))
        u = tf.random.normal((w.shape[0], 1))
        for _ in range(self.n_iter):
            v = tf.matmul(w, u, transpose_a=True)
            v /= tf.norm(v)

            u = tf.matmul(w, v)
            u /= tf.norm(u)

        spec_norm = tf.matmul(u, tf.matmul(w, v), transpose_a=True)
        return input_weights/spec_norm

In [18]:
class SimplePooling(layers.Layer):
    def __init__(self, ksize=2,kk=2):
        super(SimplePooling, self).__init__()
        self.ksize = ksize
        self.kk = kk

    def call(self, inputs):
        k_size=self.ksize
        channel = inputs.shape[3]
        x_patches = tf.image.extract_patches(inputs,
                        sizes=[1,k_size,k_size,1],
                        strides=[1,k_size,k_size,1],
                        rates=[1,1,1,1],
                        padding='VALID')

        return tf.concat([tf.reduce_mean(tf.math.top_k(x_patches[:,:,:,c::channel],k=self.kk).values,keepdims=True, axis=-1) for c in range(channel)], axis=-1)

In [19]:
class EuclideanDistanceLayer(Layer):
    def __init__(self, **kwargs):
        super(EuclideanDistanceLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # Assuming inputs is a list of two feature maps with the same shape
        feature_map1, feature_map2 = inputs

        # Calculate the pixel-wise Euclidean distance
        squared_diff = tf.square(feature_map1 - feature_map2)
        sum_squared_diff = tf.reduce_sum(squared_diff, axis=-1, keepdims=True)
        euclidean_distance = tf.sqrt(sum_squared_diff)

        return euclidean_distance

In [20]:
@tf.keras.utils.register_keras_serializable(package="Custom")
class SelfAttention1(Layer):
    def __init__(self, groups=16, **kwargs):  # Add groups parameter
        super(SelfAttention, self).__init__(**kwargs)
        self.groups = groups

    def build(self, input_shape):
        # Modified to handle the extra dimension for groups
        n, h, w, z, c_per_group = input_shape # c_per_group = C/(2G)
        self.n_feats = h * w
        self.conv_theta = Conv2D(c_per_group // 2, 1, padding='same', name='Conv_Theta')
        self.conv_phi = Conv2D(c_per_group // 2, 1, padding='same', name='Conv_Phi')
        self.conv_g = Conv2D(c_per_group // 2, 1, padding='same', name='Conv_G')
        self.conv_attn_g = Conv2D(c_per_group, 1, padding='same', name='Conv_AttnG')
        self.sigma = self.add_weight(shape=[1], initializer='zeros', trainable=True, name='sigma')

    def call(self, x):
        n, h, w, z, c_per_group = x.shape  # c_per_group = C/(2G)

        # Reshape for group-wise attention: [B, H, W, G, C/(2G)] -> [B*G, H, W, C/(2G)]
        x = tf.reshape(x, [-1, h, w, c_per_group])

        theta = self.conv_theta(x)
        theta = tf.reshape(theta, (-1, self.n_feats, theta.shape[-1]))

        phi = self.conv_phi(x)
        phi = tf.nn.max_pool2d(phi, ksize=2, strides=2, padding='VALID')
        phi = tf.reshape(phi, (-1, self.n_feats // 4, phi.shape[-1])) # Changed // 4 to // 16 to match theta shape

        attn = tf.matmul(theta, phi, transpose_b=True)
        attn = tf.nn.softmax(attn)

        g = self.conv_g(x)
        g = tf.nn.max_pool2d(g, ksize=2, strides=2, padding='VALID')
        g = tf.reshape(g, (-1, self.n_feats // 4, g.shape[-1])) # Changed // 4 to // 16 to match theta shape

        attn_g = tf.matmul(attn, g)
        attn_g = tf.reshape(attn_g, (-1, h, w, attn_g.shape[-1]))  # Back to [B*G, H, W, C/(2G)]
        attn_g = self.conv_attn_g(attn_g)

        # Reshape back to original group structure: [B*G, H, W, C/(2G)] -> [B, H, W, G, C/(2G)]
        attn_g = tf.reshape(attn_g, [-1, h, w, self.groups, c_per_group])

        x = tf.reshape(x, [-1, h, w, self.groups, c_per_group])  # Original input reshaped

        output = x + self.sigma * attn_g

        return output

    def get_config(self):
        config = super().get_config()
        config.update({"groups": self.groups})  # Include groups in config
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [21]:
@tf.keras.utils.register_keras_serializable(package="Custom")
class SALayer(Layer):
    def __init__(self, channel, groups=16, **kwargs):
        super(SALayer, self).__init__(**kwargs)
        self.groups = groups
        self.channel = channel
        self.group_channels = channel // groups  # Channels per group

    def build(self, input_shape):
        # Channel Attention: Global Average Pooling + Learnable Parameters
        self.avg_pool = GlobalAveragePooling2D()
        self.cweight = self.add_weight(
            shape=(1, 1, 1, self.channel // 2),
            initializer="zeros",
            trainable=True,
            name="cweight",
        )
        self.cbias = self.add_weight(
            shape=(1, 1, 1, self.channel // 2),
            initializer="ones",
            trainable=True,
            name="cbias",
        )

        # Spatial Attention: Self-Attention Mechanism
        self.spatial_attention = SelfAttention1()

        super(SALayer, self).build(input_shape)

    def channel_shuffle(self, x, groups):
        """Shuffle channels to promote cross-group information flow."""
        batch, height, width, channels = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]
        x = tf.reshape(x, [batch, height, width, groups, -1])  # [B, H, W, G, C/G]
        x = tf.transpose(x, perm=[0, 1, 2, 4, 3])  # [B, H, W, C/G, G]
        x = tf.reshape(x, [batch, height, width, channels])  # [B, H, W, C]
        return x

    def call(self, x):
        # Input shape: [B, H, W, C]
        batch, height, width, channels = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]

        # Reshape into groups: [B, H, W, G, C/G]
        x = tf.reshape(x, [batch, height, width, self.groups, self.group_channels])

        # Split into two branches: [B, H, W, G, C/(2G)]
        x_0, x_1 = tf.split(x, num_or_size_splits=2, axis=-1)

        # Channel Attention Branch
        # Reshape x_0 before applying GlobalAveragePooling2D
        x_0_reshaped = tf.reshape(x_0, [batch, height, width, self.groups * (self.group_channels // 2)])
        xn = self.avg_pool(x_0_reshaped)  # [B, G, C/(2G)]
        xn = self.cweight * xn + self.cbias  # Apply learnable weights and bias
        xn = tf.reshape(xn, [batch, 1, 1, self.groups, self.group_channels // 2])  # [B, 1, 1, G, C/(2G)]
        xn = tf.sigmoid(xn)  # Sigmoid activation
        x_0 = x_0 * xn  # Apply channel attention

        # Spatial Attention Branch
        xs = self.spatial_attention(x_1)  # [B, H, W, G, C/(2G)]

        # Concatenate branches: [B, H, W, G, C/G]
        out = tf.concat([x_0, xs], axis=-1)

        # Reshape back to original shape: [B, H, W, C]
        out = tf.reshape(out, [batch, height, width, channels])

        # Channel Shuffle to mix information across groups
        out = self.channel_shuffle(out, self.groups)

        return out

    def get_config(self):
        config = super().get_config()
        config.update({"channel": self.channel, "groups": self.groups})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [22]:
class SelfAttention(Layer):
    def __init__(self, ksize=4, stride=4, ratio=8, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.ksize = ksize
        self.stride = stride
        self.ratio = ratio

    def build(self, input_shape):
        n, h, w, c = input_shape
        self.n_feats = h * w
        # Reduced channel dimension
        reduced_c = c // self.ratio
        self.conv_theta = Conv2D(reduced_c, 1, padding='same')
        self.conv_phi = Conv2D(reduced_c, 1, padding='same')
        self.conv_g = Conv2D(c, 1, padding='same')
        self.conv_gc = Conv2D(c, 1, padding='same')

        # Gating weights
        self.gate_sa = self.add_weight(name="gate_sa", shape=(1,), initializer="zeros", trainable=True)
        self.gate_ca = self.add_weight(name="gate_ca", shape=(1,), initializer="zeros", trainable=True)

    def call(self, x):
        n, h, w, c = x.shape
        theta = self.conv_theta(x)
        theta = tf.reshape(theta, (-1, self.n_feats, theta.shape[-1]))

        phi = self.conv_phi(x)
        phi_s = tf.nn.max_pool2d(phi, self.ksize, self.stride, padding='VALID')
        # The error was in this line. We need to divide n_feats by (ksize * stride)
        # and multiply stride because of the maxpool
        phi_s = tf.reshape(phi_s, (-1, self.n_feats // (self.ksize * self.stride), phi_s.shape[-1]))

        attn = tf.matmul(theta, phi_s, transpose_b=True)
        attn = tf.nn.softmax(attn)

        g = self.conv_g(x)
        g = tf.nn.max_pool2d(g, self.ksize, self.stride, padding='VALID')
        # Same correction as in line 30
        g = tf.reshape(g, (-1, self.n_feats // (self.ksize * self.stride), g.shape[-1]))

        attn_g = tf.matmul(attn, g)
        attn_g = tf.reshape(attn_g, (-1, h, w, attn_g.shape[-1]))

        # Compute attention for CAM (Channel Attention Module)
        theta_c = tf.transpose(theta, perm=[0, 2, 1])
        phi = tf.reshape(phi, (-1, self.n_feats, phi.shape[-1]))

        attn_c = tf.matmul(theta_c, phi)
        attn_c = tf.nn.softmax(attn_c)

        g_c = self.conv_gc(x)
        g_c = tf.reshape(g_c, (-1, self.n_feats*self.ratio, c//self.ratio))

        attn_c = tf.matmul(g_c, attn_c)
        attn_c = tf.reshape(attn_c, (-1, h, w, c))
        #attn_c = self.conv_attn_c(attn_c)

        output = x + self.gate_sa * attn_g + self.gate_ca * attn_c

        return output
    def get_config(self):
        config = super().get_config()
        config.update({"ksize": self.ksize, "stride": self.stride, "ratio": self.ratio})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

## DEFINITION OF FUNCTIONS FOR THE MODEL


*   CSA attention block---> GRN+Droppath class

*   CBAM block---> Spatial and channel
*   Convolutional blocks (Inverted, Normal and Residual)


*   Shared Feature Information modules 1 (SFIM1, SFIM1_0)--->Attention class

*   Shared Feature Information modules 2 (SFIM2)---> SPA class

*   Focal Feature Upsampling modules 1 (FFUM1)---> Channel attention class









In [32]:
def res_conv_block(x, kernelsize, filters, dropout, batchnorm=True):
    shortcut = x  # Identity connection
    c= x.shape[3]

    # Main path
    res = layers.Conv2D(filters, kernelsize, strides=1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    if batchnorm:
        res = layers.BatchNormalization()(res)
    res = layers.ReLU()(res)
    if dropout > 0:
        res = layers.Dropout(dropout)(res)

    res = layers.Conv2D(filters, kernelsize, strides=1, padding='same', use_bias=False, kernel_initializer='he_normal')(res)
    if batchnorm:
        res = layers.BatchNormalization()(res)
    if dropout > 0:
        res = layers.Dropout(dropout)(res)


    # Shortcut path (if needed)
    if c != filters:
        shortcut = layers.Conv2D(filters, kernel_size=1, strides=1, padding='same', use_bias=False, kernel_initializer='he_normal',)(x)

    # Residual connection
    output = layers.add([shortcut, res])

    return output

In [33]:
#convolutional block
def conv_block(x, kernelsize, filters,  batchnorm=False):
    conv = Conv2D(filters, (kernelsize, kernelsize),  padding="same")(x)
    if batchnorm is True:
        conv = BatchNormalization(axis=3)(conv)
    conv = Activation("relu")(conv)
    return conv

#convolutional block1
def conv_block1(x, kernelsize, filters, dropout,  batchnorm=False):
    conv = Conv2D(filters, (kernelsize, kernelsize), padding="same", kernel_initializer='he_normal')(x)
    if batchnorm is True:
        conv = BatchNormalization(axis=3)(conv)
    if dropout > 0:
        conv = Dropout(dropout, seed=42)(conv)
    conv = Conv2D(filters, (kernelsize, kernelsize), padding="same", kernel_initializer='he_normal')(conv)
    if batchnorm is True:
        conv = BatchNormalization(axis=3)(conv)
    conv = Activation("relu")(conv)
    return conv

In [34]:
def inverted_residual_block(inputs, expansion_factor, output_channels, stride):
    input_channels = inputs.shape[-1]
    x = inputs

    # Expand phase
    if expansion_factor != 1:
        x = layers.Conv2D(input_channels * expansion_factor, 1, padding='same', use_bias=True, activation=keras.activations.hard_silu)(x)

    # Depthwise Convolution
    x = layers.DepthwiseConv2D(3, strides=stride,  padding='same', use_bias=True, activation=keras.activations.hard_silu)(x)

    # Linear bottleneck
    x = layers.Conv2D(output_channels, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride == 1 and input_channels == output_channels:
        x = layers.Add()([inputs, x])

    return x

In [35]:
def ChannelWiseAttention(inputs, reduction_ratio=8):
    """
    Channel-wise Attention Module using Squeeze-and-Excitation (SE) block.

    Args:
        inputs: Tensor of shape (batch_size, H, W, C).
        reduction_ratio: Integer, reduction ratio for bottleneck transformation.

    Returns:
        Tensor of same shape as inputs, with enhanced channel attention.
    """
    channel_dim = inputs.shape[-1]  # Number of channels (C)

    # Squeeze: Global Average Pooling
    squeezed = GlobalAveragePooling2D()(inputs)  # Shape: (batch_size, C)

    # Reshape for dense layer processing
    squeezed = Reshape((1, 1, channel_dim))(squeezed)  # Shape: (batch_size, 1, 1, C)

    # Fully connected layers for channel-wise attention
    reduced = Dense(channel_dim // reduction_ratio, activation='relu', use_bias=False, kernel_initializer='he_normal')(squeezed)  # Bottleneck layer
    attention_weights = Dense(channel_dim, activation='sigmoid', use_bias=False, kernel_initializer='he_normal')(reduced)  # Final channel weights

    # Scale original features with computed attention weights
    output = Multiply()([inputs, attention_weights])  # Element-wise multiplication

    return output

In [36]:
def SFIM1_0(F1_pre, F2_post, F_DEM, name):
    channel = F1_pre.shape[-1]  # Number of channels (C)

    mi = EuclideanDistanceLayer()([F1_pre, F2_post])
    # Apply sigmoid activation to get attention map ai
    ai = Activation('sigmoid')(mi)
    x = tf.keras.layers.Concatenate(axis=-1)([F1_pre, F2_post])
    x = Conv2D(filters= channel, kernel_size=(1, 1),  padding='same', kernel_initializer='he_normal')(x)
    # Elementwise multiplication of Gi and ai
    diff_features = tf.keras.layers.Multiply()([x, ai])

    #diff_features = layers.Concatenate(axis=-1)([F1_pre, F2_post])
    # Concatenate DEM and difference features
    fused_features = layers.Concatenate(axis=-1)([diff_features, F_DEM])
    # Apply channel-wise attention to enhance important features
    channel_features = ChannelWiseAttention(fused_features)
    res = layers.Conv2D(channel, kernel_size = 3, strides=1, padding='same', kernel_initializer='he_normal')(channel_features)
    res = layers.BatchNormalization()(res)
    res = layers.ReLU()(res)
    return res

In [37]:
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    shared_layer_one = Dense(channel//ratio,
                             activation='relu',
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
    shared_layer_two = Dense(channel,
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1,1,channel))(avg_pool)
    # Changed _keras_shape to shape
    assert avg_pool.shape[1:] == (1,1,channel)
    avg_pool = shared_layer_one(avg_pool)
    # Changed _keras_shape to shape
    assert avg_pool.shape[1:] == (1,1,channel//ratio)
    avg_pool = shared_layer_two(avg_pool)
    # Changed _keras_shape to shape
    assert avg_pool.shape[1:] == (1,1,channel)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1,1,channel))(max_pool)
    # Changed _keras_shape to shape
    assert max_pool.shape[1:] == (1,1,channel)
    max_pool = shared_layer_one(max_pool)
    # Changed _keras_shape to shape
    assert max_pool.shape[1:] == (1,1,channel//ratio)
    max_pool = shared_layer_two(max_pool)
    # Changed _keras_shape to shape
    assert max_pool.shape[1:] == (1,1,channel)

    cbam_feature = Add()([avg_pool,max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)

    return multiply([input_feature, cbam_feature])

In [38]:
def SFIM2(input_tensor, filters, name, dilation_rates=[6, 12, 18]):

    # 1x1 Convolution (Without dilation)
    conv_1x1 = Conv2D(filters, (1, 1), padding='same', use_bias=False, kernel_initializer=keras.initializers.HeNormal())(input_tensor)
    conv_1x1 = BatchNormalization()(conv_1x1)
    conv_1x1 = Activation('relu')(conv_1x1)

    # Atrous Convolutions with different dilation rates
    atrous_convs = []
    for rate in dilation_rates:
        x = Conv2D(filters, (3, 3), padding='same', dilation_rate=rate, use_bias=False, kernel_initializer=keras.initializers.HeNormal())(input_tensor)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        atrous_convs.append(x)

    # Global Average Pooling followed by 1x1 Convolution
    global_avg = GlobalAveragePooling2D()(input_tensor)
    global_avg = layers.Reshape((1, 1, global_avg.shape[-1]))(global_avg)
    global_avg = Conv2D(filters, (1, 1), padding='same', use_bias=True)(global_avg)
    global_avg = BatchNormalization()(global_avg)
    global_avg = Activation('relu')(global_avg)
    global_avg = UpSampling2D(size=(input_tensor.shape[1], input_tensor.shape[2]), interpolation="bilinear")(global_avg)

    # Concatenate all outputs
    x = Concatenate(axis=-1)([conv_1x1] + atrous_convs + [global_avg])
    x = channel_attention(x)

    # Final 1x1 convolution
    output = tf.keras.layers.Conv2D(filters, (1, 1), padding='same', use_bias=False, kernel_initializer=keras.initializers.HeNormal(), activation='relu',  name=name)(x)
    return output

In [39]:
def UpSample(x, filters, interpolation='bilinear'):
    x = layers.UpSampling2D(size=2, interpolation=interpolation)(x)
    x = layers.Conv2D(
        filters, kernel_size=3, padding="same", use_bias=False, kernel_initializer='he_normal'
    )(x)
    return x

In [40]:
#Version 2 of FFUM1
def FFUM1(X_i, X_i_plus_1, filters , name):

    # Step 1: Apply two deconvolutions with 1x1 convolution and batch normalization
    conv1 = UpSample(X_i_plus_1, filters)

    # Step 2: Concatenate the feature maps
    S1 = Concatenate(axis=-1)([X_i, conv1])

    # Attention map generation
    psi = relu((Add()([X_i, conv1])))  # Add and apply ReLU
    psi = Conv2D(1, kernel_size=3, strides=1, padding='same', use_bias=False, kernel_initializer='he_normal')(psi)
    psi = BatchNormalization()(psi)
    psi = tf.keras.layers.Activation('sigmoid')(psi)  # Apply sigmoid

    # Apply attention and return
    attended_x = Multiply()([S1, psi])

    # Apply a 3x3 convolution
    output = res_conv_block(attended_x, kernelsize=3, filters=filters, dropout=0.3, batchnorm=True)

    return output

## MAIN BRANCH OF THE SIAMESE MODEL

In [41]:
#Main branch encoder
def Main_branch_CSAencoder(filtersFirstLayer, input_shape, batchnorm=True):

    # Define input within the function
    input_tensor = Input(shape=input_shape)
    # Initial stem Convolution Layer
    F1 = layers.Conv2D(filtersFirstLayer, 3, strides=(1, 1), padding='same', use_bias=False)(input_tensor)
    F1 = Dropout(0.1)(F1)
    F1 = layers.Conv2D(filtersFirstLayer, 3, strides=(1, 1), padding='same', use_bias=False)(F1)
    F1 = layers.BatchNormalization()(F1)
    F1 = layers.ReLU()(F1)

    # Second layer encoder

    F2 = inverted_residual_block(F1, expansion_factor=4, output_channels=filtersFirstLayer, stride=1)
    F2 = MaxPooling2D(pool_size=(2, 2), name='F2_layer')(F2)
    F2_att = SelfAttention(ksize=4, stride=4, ratio=4) (F2)
    #print(F2.shape)

    # Third layer encoder

    F3 = inverted_residual_block(F2, expansion_factor=4, output_channels=filtersFirstLayer*2, stride=1)
    for _ in range(0):
        F3 = inverted_residual_block(F3, expansion_factor=4, output_channels=filtersFirstLayer*2, stride=1)
    F3 = MaxPooling2D(pool_size=(2, 2), name='F3_layer')(F3)
    F3_att = SelfAttention(ksize=2, stride=2, ratio=4) (F3)
    #F3 = SALayer(filtersFirstLayer*2)(F3)
    #print(F3.shape)

    # Fourth layer encoder

    F4 = inverted_residual_block(F3, expansion_factor=4, output_channels=filtersFirstLayer*4, stride=1)
    for _ in range(1):
        F4 = inverted_residual_block(F4, expansion_factor=4, output_channels=filtersFirstLayer*4, stride=1)
    F4 = MaxPooling2D(pool_size=(2, 2), name='F4_layer')(F4)
    F4_att = SelfAttention(ksize=1, stride=1, ratio=4) (F4)
    #F4 = SALayer(filtersFirstLayer*4)(F4)
    #print(F4.shape)

    # Fifth layer encoder

    F5 = inverted_residual_block(F4, expansion_factor=4, output_channels=filtersFirstLayer*8, stride=1)
    for _ in range(1):
        F5 = inverted_residual_block(F5, expansion_factor=4, output_channels=filtersFirstLayer*8, stride=1)
    F5 = MaxPooling2D(pool_size=(2, 2), name='F5_layer')(F5)
    F5_att = SelfAttention(ksize=1, stride=1, ratio=4) (F5)
    #F5 = SALayer(filtersFirstLayer*8)(F5)
    #print(F5.shape)

    F6 = inverted_residual_block(F5, expansion_factor=4, output_channels=filtersFirstLayer*16, stride=1)
    for _ in range(1):
        F6 = inverted_residual_block(F6, expansion_factor=4, output_channels=filtersFirstLayer*16, stride=1)
    F6 = MaxPooling2D(pool_size=(2, 2), name='F6_layer')(F6)

    model = Model(inputs= input_tensor, outputs=[F1, F2_att, F3_att, F4_att, F5_att, F6])
    return model

## MAIN FULL MODEL

In [42]:
size1 = 256
img_bands1= 11
size2 = 256
img_bands2 = 6

In [43]:
def Siamese_CSA_2(filtersFirstLayer, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2),batchnorm=True):

    #Define the inputs
    input_1 = Input(shape=input_size1, name='input_1')  # Pre-event optical
    input_2 = Input(shape=input_size2, name='input_2')  # Post-event optical
    input_3 = Input(shape=input_size3, name='input_3')  # Topographic

    # Define shared encoder with correct input shape
    encoder_model = Main_branch_CSAencoder(filtersFirstLayer, input_shape = input_size1)

    # Apply encoder to both concatenated inputs
    Features1 = encoder_model(input_1)
    Features2 = encoder_model(input_2)

    # Unpacking encoder outputs
    F1_1, F2_1, F3_1, F4_1, F5_1, F6_1 = Features1
    F1_2, F2_2, F3_2, F4_2, F5_2, F6_2 = Features2

    #1st layer of the second branch
    G1 =  inverted_residual_block(input_3, expansion_factor=4, output_channels=filtersFirstLayer, stride=1)

    sfi1_0 = SFIM1_0(F1_1, F1_2, G1, name='SFI1_0')
    #print(sfi1_0.shape)

    # Second layer encoder

    # Inverted Residual Blocks
    G2 = inverted_residual_block(G1, expansion_factor=4, output_channels=filtersFirstLayer, stride=1)
    G2 = MaxPooling2D(pool_size=(2, 2), name='G2_layer')(G2)
    #print(G2.shape)

    #Shared Feature information module 1
    sfi1_1 = SFIM1_0(F2_1, F2_2, G2, name='SFI1_1')
    #print(sfi1_1.shape)

    # Third layer encoder

    # Inverted Residual Blocks
    G3 = inverted_residual_block(G2, expansion_factor=4, output_channels=filtersFirstLayer*2, stride=1)
    G3 = MaxPooling2D(pool_size=(2, 2), name='G3_layer')(G3)
    #print(G3.shape)

    #Shared Feature information module 1
    sfi1_2 = SFIM1_0(F3_1, F3_2, G3, name='SFI1_2')
    #print(sfi1_2.shape)

    # Fourth layer encoder

    # Inverted Residual Blocks
    G4 = inverted_residual_block(G3, expansion_factor=4, output_channels=filtersFirstLayer*4, stride=1)
    G4 = MaxPooling2D(pool_size=(2, 2), name='G4_layer')(G4)
    #print(G4.shape)

    #Shared Feature information module 1
    sfi1_3 = SFIM1_0(F4_1, F4_2, G4, name='SFI1_3')
    #print(sfi1_3.shape)

    # Fifth layer encoder

    # Inverted Residual Blocks
    G5 = inverted_residual_block(G4, expansion_factor=4, output_channels=filtersFirstLayer*8, stride=1)
    G5 = MaxPooling2D(pool_size=(2, 2), name='G5_layer')(G5)
    #print(G5.shape)

    #Shared Feature information module 1
    sfi1_4 = SFIM1_0(F5_1, F5_2, G5, name='SFI1_4')
    #print(sfi1_4.shape)

    # sixth layer encoder

    # Inverted Residual Blocks
    G6 = inverted_residual_block(G5, expansion_factor=4, output_channels=filtersFirstLayer*16, stride=1)
    G6 = MaxPooling2D(pool_size=(2, 2), name='G6_layer')(G6)
    #print(G6.shape)

    # bottlneck layer encoder
    #fuse  = layers.Add(name='fuse')([F6_1, F6_2])
    #sfi1_5 = Concatenate(axis=-1)([fuse, G6])
    sfi1_5 = Concatenate(axis=-1)([F6_1, F6_2, G6])
    sfi1_5 = conv_block1(sfi1_5, kernelsize=3, filters=filtersFirstLayer*16, dropout=0.3, batchnorm=False)
    #print(sfi1_5.shape)

    #Shared Feature information module 2
    sfi2 = SFIM2(sfi1_5, filters=filtersFirstLayer*16, name='SFI2')
    #print(sfi2.shape)

    #1st layer Decoder
    Y_2 = FFUM1(sfi1_4, sfi2, filters=filtersFirstLayer*8, name='FFUM1_1')
    #print(Y_2.shape)

    #2nd layer Decoder
    Y_3 = FFUM1(sfi1_3, Y_2, filters=filtersFirstLayer*4, name='FFUM1_2')
    #print(Y_3.shape)

    #3rd layer Decoder
    Y_4 = FFUM1(sfi1_2, Y_3, filters=filtersFirstLayer*2, name='FFUM1_3')

    #4th layer Decoder
    Y_5 = FFUM1(sfi1_1, Y_4, filters=filtersFirstLayer, name='FFUM1_4')

    #Classification Layer
    Y_6 = UpSample(Y_5, filters=filtersFirstLayer)
    merge = Concatenate(axis=-1)([Y_6, sfi1_0])
    Y_6 = conv_block1(merge, kernelsize=3, filters=filtersFirstLayer, dropout=0.3, batchnorm=True)

    #print(Y_4.shape)
    Final = conv_block(Y_6, kernelsize=1, filters=1, batchnorm=False)
    Final = Activation('sigmoid')(Final)
    #print(Final.shape)

    return Model(inputs=[input_1, input_2, input_3], outputs=Final)

In [48]:
import psutil
# fix random seed for reproducibility
np.random.seed(42)
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']

# Loss function. We assign the variable called "loss" which takes the dice_loss loss function.
loss= loss

# Number of filters. We set a range of the number of filters for the convolutional layers.
filters = 64

lr = 10e-6

# Dictionary that will save the results. We first make an empty dictionary so that later we can save/store all the important configurations that we experimented with and report at the end in nice Excel CSVs and Plots.
dic = {}

# Hyperparameters. These are the keys where the associated information for each hyperparameter will be saved.
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []

GASA = Siamese_CSA_2(filtersFirstLayer=filters, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2),batchnorm=True)
optimizer = Adam(learning_rate=lr)
GASA.compile(optimizer = optimizer, loss = loss, metrics = metrics)
# load the last saved weight from the training
GASA.load_weights('/content/drive/MyDrive/landslide/Selection_model/Model_2_withAda/Siamese2_CSA_size_256_filters_64_batch_size_4_lr_0.0001.weights.h5')
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    X_test3_roi = X_test3[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test2_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = GASA.evaluate( [X_test1_roi, X_test2_roi, X_test3_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("GASA")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])
# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_GASA_by_ROI.csv', index = False)

Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 25s 8s/step - IoU: 0.2885 - accuracy: 0.9475 - f1-score: 0.4465 - loss: 0.8134 - precision: 0.4305 - recall: 0.4762
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - IoU: 0.2466 - accuracy: 0.9370 - f1-score: 0.3956 - loss: 0.8807 - precision: 0.2501 - recall: 0.9454
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - IoU: 0.2496 - accuracy: 0.9806 - f1-score: 0.3994 - loss: 0.9348 - precision: 0.3704 - recall: 0.4334
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - IoU: 0.2884 - accuracy: 0.9881 - f1-score: 0.4477 - loss: 0.9576 - precision: 0.3746 - recall: 0.5565
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 18s 18s/step - IoU: 0.4302 - accuracy: 0.9448 - f1-score: 0.6016 - loss: 0.7376 - precision: 0.6025 - recall: 0.6007


In [49]:
import psutil
# fix random seed for reproducibility
np.random.seed(42)
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']

# Loss function. We assign the variable called "loss" which takes the dice_loss loss function.
loss= loss

# Number of filters. We set a range of the number of filters for the convolutional layers.
filters = 64

lr = 10e-6

# Dictionary that will save the results. We first make an empty dictionary so that later we can save/store all the important configurations that we experimented with and report at the end in nice Excel CSVs and Plots.
dic = {}

# Hyperparameters. These are the keys where the associated information for each hyperparameter will be saved.
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []

GASA2 = Siamese_CSA_2(filtersFirstLayer=filters, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2),batchnorm=True)
optimizer = Adam(learning_rate=lr)
GASA2.compile(optimizer = optimizer, loss = loss, metrics = metrics)
# load the last saved weight from the training
GASA2.load_weights('/content/drive/MyDrive/landslide/Selection_model/Model_2_withoutAda/Siamese2_CSA_size_256_filters_64_batch_size_4_lr_0.0001.weights.h5')
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    X_test3_roi = X_test3[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test2_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = GASA2.evaluate( [X_test1_roi, X_test2_roi, X_test3_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("GASA2")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])
# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_GASA2_by_ROI.csv', index = False)

Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 25s 9s/step - IoU: 0.2645 - accuracy: 0.9248 - f1-score: 0.4164 - loss: 0.8007 - precision: 0.3193 - recall: 0.6243
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - IoU: 0.1748 - accuracy: 0.9034 - f1-score: 0.2976 - loss: 0.8843 - precision: 0.1768 - recall: 0.9384
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - IoU: 0.2660 - accuracy: 0.9792 - f1-score: 0.4202 - loss: 0.9315 - precision: 0.3587 - recall: 0.5072
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - IoU: 0.3183 - accuracy: 0.9881 - f1-score: 0.4829 - loss: 0.9555 - precision: 0.3883 - recall: 0.6384
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - IoU: 0.4112 - accuracy: 0.9314 - f1-score: 0.5828 - loss: 0.7273 - precision: 0.5041 - recall: 0.6906


In [46]:
import psutil
# fix random seed for reproducibility
np.random.seed(42)
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']

# Loss function. We assign the variable called "loss" which takes the dice_loss loss function.
loss= loss

# Number of filters. We set a range of the number of filters for the convolutional layers.
filters = 64

lr = 10e-6

# Dictionary that will save the results. We first make an empty dictionary so that later we can save/store all the important configurations that we experimented with and report at the end in nice Excel CSVs and Plots.
dic = {}

# Hyperparameters. These are the keys where the associated information for each hyperparameter will be saved.
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []

GASA1 = Siamese_CSA_2(filtersFirstLayer=filters, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2),batchnorm=True)
optimizer = Adam(learning_rate=lr)
GASA1.compile(optimizer = optimizer, loss = loss, metrics = metrics)
# load the last saved weight from the training
GASA1.load_weights('/content/drive/MyDrive/landslide/Selection_model/Model_2_withAda_AugDiff/Siamese2_CSA_size_256_filters_64_batch_size_4_lr_0.0001.weights.h5')
# Get unique ROI values
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    X_test3_roi = X_test3[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test2_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = GASA1.evaluate( [X_test1_roi, X_test2_roi, X_test3_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("GASA1")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])
# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_GASA1_by_ROI.csv', index = False)

Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 24s 8s/step - IoU: 0.2559 - accuracy: 0.9521 - f1-score: 0.4068 - loss: 0.8249 - precision: 0.4983 - recall: 0.3443
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - IoU: 0.2874 - accuracy: 0.9521 - f1-score: 0.4465 - loss: 0.8828 - precision: 0.2984 - recall: 0.8859
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - IoU: 0.3073 - accuracy: 0.9849 - f1-score: 0.4701 - loss: 0.9337 - precision: 0.4911 - recall: 0.4508
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - IoU: 0.2311 - accuracy: 0.9894 - f1-score: 0.3755 - loss: 0.9627 - precision: 0.3839 - recall: 0.3675
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - IoU: 0.3602 - accuracy: 0.9441 - f1-score: 0.5296 - loss: 0.7576 - precision: 0.6356 - recall: 0.4539


## MODEL SIAMESE SWINUNET WITH DEPTH = 5 AND MLP=1024

In [50]:
!pip install keras-unet-collection -q -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.5 MB/s eta 0:00:00


In [51]:
from __future__ import absolute_import

from keras_unet_collection.layer_utils import *
from keras_unet_collection.transformer_layers import patch_extract, patch_embedding, patch_merging, patch_expanding, window_partition, window_reverse, drop_path, Mlp

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
import numpy as np
import tensorflow as tf
from tensorflow.nn import depth_to_space
from tensorflow.image import extract_patches
from tensorflow.keras.layers import Conv2D, Layer, Dense, Embedding, Dropout, Conv2D, LayerNormalization
from tensorflow.keras.activations import softmax, sigmoid

In [52]:
class WindowAttention(tf.keras.layers.Layer):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None,
                 attn_drop=0, proj_drop=0., name='swin_atten', **kwargs):
        super(WindowAttention, self).__init__(**kwargs)

        self.dim = dim # number of input dimensions
        self.window_size = window_size # size of the attention window
        self.num_heads = num_heads # number of self-attention heads
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.attn_drop = attn_drop
        self.proj_drop = proj_drop

        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5 # query scaling factor

        self.prefix = name

        # Layers
        self.qkv = Dense(dim * 3, use_bias=qkv_bias, name='{}_attn_qkv'.format(self.prefix))
        self.attn_drop = Dropout(attn_drop)
        self.proj = Dense(dim, name='{}_attn_proj'.format(self.prefix))
        self.proj_drop = Dropout(proj_drop)

    def get_config(self):
        config = super().get_config().copy()
        config.update({
            'dim':self.dim,
            'window_size':self.window_size,
            'num_heads':self.num_heads,
            'qkv_bias':self.qkv_bias,
            'qk_scale':self.qk_scale,
            'attn_drop':self.attn_drop,
            'proj_drop':self.proj_drop,
            'name':self.prefix
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def build(self, input_shape):

        # zero initialization
        num_window_elements = (2*self.window_size[0] - 1) * (2*self.window_size[1] - 1)
        self.relative_position_bias_table = self.add_weight(
            name='{}_attn_pos'.format(self.prefix),
            shape=(num_window_elements, self.num_heads),
            initializer=tf.initializers.Zeros(),
            trainable=True
        )


        # Indices of relative positions
        coords_h = np.arange(self.window_size[0])
        coords_w = np.arange(self.window_size[1])
        coords_matrix = np.meshgrid(coords_h, coords_w, indexing='ij')
        coords = np.stack(coords_matrix)
        coords_flatten = coords.reshape(2, -1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.transpose([1, 2, 0])
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)

        # convert to the tf variable
        with tf.init_scope():  # <-- Add this line
            self.relative_position_index = tf.Variable(
                initial_value=tf.convert_to_tensor(relative_position_index),
                trainable=False,
                name='{}_attn_pos_ind'.format(self.prefix)
            )

        self.built = True

    def call(self, x, mask=None):

        # Get input tensor static shape
        _, N, C = x.get_shape().as_list()
        head_dim = C//self.num_heads

        x_qkv = self.qkv(x)
        x_qkv = tf.reshape(x_qkv, shape=(-1, N, 3, self.num_heads, head_dim))
        x_qkv = tf.transpose(x_qkv, perm=(2, 0, 3, 1, 4))
        q, k, v = x_qkv[0], x_qkv[1], x_qkv[2]

        # Query rescaling
        q = q * self.scale

        # multi-headed self-attention
        k = tf.transpose(k, perm=(0, 1, 3, 2))
        attn = (q @ k)

        # Shift window
        num_window_elements = self.window_size[0] * self.window_size[1]
        relative_position_index_flat = tf.reshape(self.relative_position_index, shape=(-1,))
        relative_position_bias = tf.gather(self.relative_position_bias_table, relative_position_index_flat)
        relative_position_bias = tf.reshape(relative_position_bias, shape=(num_window_elements, num_window_elements, -1))
        relative_position_bias = tf.transpose(relative_position_bias, perm=(2, 0, 1))
        attn = attn + tf.expand_dims(relative_position_bias, axis=0)

        if mask is not None:
            nW = mask.get_shape()[0]
            mask_float = tf.cast(tf.expand_dims(tf.expand_dims(mask, axis=1), axis=0), tf.float32)
            attn = tf.reshape(attn, shape=(-1, nW, self.num_heads, N, N)) + mask_float
            attn = tf.reshape(attn, shape=(-1, self.num_heads, N, N))
            attn = softmax(attn, axis=-1)
        else:
            attn = softmax(attn, axis=-1)

        # Dropout after attention
        attn = self.attn_drop(attn)

        # Merge qkv vectors
        x_qkv = (attn @ v)
        x_qkv = tf.transpose(x_qkv, perm=(0, 2, 1, 3))
        x_qkv = tf.reshape(x_qkv, shape=(-1, N, C))

        # Linear projection
        x_qkv = self.proj(x_qkv)

        # Dropout after projection
        x_qkv = self.proj_drop(x_qkv)

        return x_qkv

In [53]:
class SwinTransformerBlock(tf.keras.layers.Layer):
    def __init__(self, dim, num_patch, num_heads, window_size=7, shift_size=0,
                 num_mlp=1024, qkv_bias=True, qk_scale=None, mlp_drop=0, attn_drop=0,
                 proj_drop=0, drop_path_prob=0, name='swin_block', **kwargs):

        super(SwinTransformerBlock, self).__init__(**kwargs)

        self.dim = dim # number of input dimensions
        self.num_patch = num_patch # number of embedded patches; a tuple of  (heigh, width)
        self.num_heads = num_heads # number of attention heads
        self.window_size = window_size # size of window
        self.shift_size = shift_size # size of window shift
        self.num_mlp = num_mlp # number of MLP nodes
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.mlp_drop = mlp_drop
        self.attn_drop = attn_drop
        self.proj_drop = proj_drop
        self.drop_path_prob = drop_path_prob

        self.prefix = name

        # Layers
        self.norm1 = LayerNormalization(epsilon=1e-5, name='{}_norm1'.format(self.prefix))
        self.attn = WindowAttention(dim, window_size=(self.window_size, self.window_size), num_heads=num_heads,
                                    qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=proj_drop, name=self.prefix)
        self.drop_path = drop_path(drop_path_prob)
        self.norm2 = LayerNormalization(epsilon=1e-5, name='{}_norm2'.format(self.prefix))
        self.mlp = Mlp([num_mlp, dim], drop=mlp_drop, name=self.prefix)

        # Assertions
        assert 0 <= self.shift_size, 'shift_size >= 0 is required'
        assert self.shift_size < self.window_size, 'shift_size < window_size is required'

        # <---!!!
        # Handling too-small patch numbers
        if min(self.num_patch) < self.window_size:
            self.shift_size = 0
            self.window_size = min(self.num_patch)

    def get_config(self):
        config = super().get_config().copy()
        config.update({
            'dim':self.dim,
            'num_patch':self.num_patch,
            'num_heads':self.num_heads,
            'window_size':self.window_size,
            'shift_size':self.shift_size,
            'num_mlp':self.num_mlp,
            'qkv_bias':self.qkv_bias,
            'qk_scale':self.qk_scale,
            'mlp_drop':self.mlp_drop,
            'attn_drop':self.attn_drop,
            'proj_drop':self.proj_drop,
            'drop_path_prob':self.drop_path_prob,
            'name':self.prefix
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def build(self, input_shape):
        if self.shift_size > 0:
            H, W = self.num_patch
            h_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))

            # attention mask
            mask_array = np.zeros((1, H, W, 1))

            ## initialization
            count = 0
            for h in h_slices:
                for w in w_slices:
                    mask_array[:, h, w, :] = count
                    count += 1
            mask_array = tf.convert_to_tensor(mask_array)

            # mask array to windows
            mask_windows = window_partition(mask_array, self.window_size)
            mask_windows = tf.reshape(mask_windows, shape=[-1, self.window_size * self.window_size])
            attn_mask = tf.expand_dims(mask_windows, axis=1) - tf.expand_dims(mask_windows, axis=2)
            attn_mask = tf.where(attn_mask != 0, -100.0, attn_mask)
            attn_mask = tf.where(attn_mask == 0, 0.0, attn_mask)
            self.attn_mask = tf.Variable(initial_value=attn_mask, trainable=False, name='{}_attn_mask'.format(self.prefix))

        else:
            self.attn_mask = None

        self.built = True

    def call(self, x):
        H, W = self.num_patch
        B, L, C = x.get_shape().as_list()

        # Checking num_path and tensor sizes
        assert L == H * W, 'Number of patches before and after Swin-MSA are mismatched.'

        # Skip connection I (start)
        x_skip = x

        # Layer normalization
        x = self.norm1(x)

        # Convert to aligned patches
        x = tf.reshape(x, shape=(-1, H, W, C))

        # Cyclic shift
        if self.shift_size > 0:
            shifted_x = tf.roll(x, shift=[-self.shift_size, -self.shift_size], axis=[1, 2])
        else:
            shifted_x = x

        # Window partition
        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = tf.reshape(x_windows, shape=(-1, self.window_size * self.window_size, C))

        # Window-based multi-headed self-attention
        attn_windows = self.attn(x_windows, mask=self.attn_mask)

        # Merge windows
        attn_windows = tf.reshape(attn_windows, shape=(-1, self.window_size, self.window_size, C))
        shifted_x = window_reverse(attn_windows, self.window_size, H, W, C)

        # Reverse cyclic shift
        if self.shift_size > 0:
            x = tf.roll(shifted_x, shift=[self.shift_size, self.shift_size], axis=[1, 2])
        else:
            x = shifted_x

        # Convert back to the patch sequence
        x = tf.reshape(x, shape=(-1, H*W, C))

        # Drop-path
        ## if drop_path_prob = 0, it will not drop
        x = self.drop_path(x)

        # Skip connection I (end)
        x = x_skip +  x

        # Skip connection II (start)
        x_skip = x

        x = self.norm2(x)
        x = self.mlp(x)
        x = self.drop_path(x)

        # Skip connection II (end)
        x = x_skip + x

        return x


In [54]:
def swin_transformer_stack(X, stack_num, embed_dim, num_patch, num_heads, window_size, num_mlp, shift_window=True, name=''):
    '''
    Stacked Swin Transformers that share the same token size.

    Alternated Window-MSA and Swin-MSA will be configured if `shift_window=True`, Window-MSA only otherwise.
    *Dropout is turned off.
    '''
    # Turn-off dropouts
    mlp_drop_rate = 0 # Droupout after each MLP layer
    attn_drop_rate = 0 # Dropout after Swin-Attention
    proj_drop_rate = 0 # Dropout at the end of each Swin-Attention block, i.e., after linear projections
    drop_path_rate = 0 # Drop-path within skip-connections

    qkv_bias = True # Convert embedded patches to query, key, and values with a learnable additive value
    qk_scale = None # None: Re-scale query based on embed dimensions per attention head # Float for user specified scaling factor

    if shift_window:
        shift_size = window_size // 2
    else:
        shift_size = 0

    for i in range(stack_num):

        if i % 2 == 0:
            shift_size_temp = 0
        else:
            shift_size_temp = shift_size

        X = SwinTransformerBlock(dim=embed_dim, num_patch=num_patch, num_heads=num_heads,
                                 window_size=window_size, shift_size=shift_size_temp, num_mlp=num_mlp, qkv_bias=qkv_bias, qk_scale=qk_scale,
                                 mlp_drop=mlp_drop_rate, attn_drop=attn_drop_rate, proj_drop=proj_drop_rate, drop_path_prob=drop_path_rate,
                                 name='name{}'.format(i))(X)
    return X

In [55]:
# Size of the tiles/patches
size1 = 256 # This line takes the value of the 3rd index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 128.
size2 = 256

# Image bands
img_bands1 = 11 # This line takes the value of the 4th index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 11.

In [56]:
def make_swin_unet_encoder(input_size=(size1,size1,img_bands1),
                           filter_num_begin=64,
                           depth=5,
                           stack_num_down=2,
                           patch_size=(4,4),
                           num_heads=[4,8,8,8,8],
                           window_size=[4,2,2,2,2],
                           num_mlp=1024,
                           shift_window=True):

    # 1) build the “base” exactly as in your swin_unet_2d_base,
    # but stop right after collecting the X_skip list.
    IN = Input(input_size, name='encoder_input')
    X = IN

    # patch extract & embed
    num_patch_x = input_size[0] // patch_size[0]
    num_patch_y = input_size[1] // patch_size[1]
    embed_dim = filter_num_begin

    X = patch_extract(patch_size)(X)
    X = patch_embedding(num_patch_x*num_patch_y, embed_dim)(X)

    X_skip = []
    # first stack
    X = swin_transformer_stack(X, stack_num_down,
                               embed_dim, (num_patch_x,num_patch_y),
                               num_heads[0], window_size[0],
                               num_mlp, shift_window,
                               name='down0')
    X_skip.append(X)

    # downsampling
    for i in range(depth-1):
        X = patch_merging((num_patch_x,num_patch_y),
                          embed_dim, name=f'down_merge{i}')(X)
        embed_dim *= 2
        num_patch_x //= 2;  num_patch_y //= 2

        X = swin_transformer_stack(X, stack_num_down,
                                   embed_dim, (num_patch_x,num_patch_y),
                                   num_heads[i+1], window_size[i+1],
                                   num_mlp, shift_window,
                                   name=f'down{i+1}')
        X_skip.append(X)

    # X_skip = [level0, level1, … level{depth-1}]
    # we will reverse so that X_skip[0] is the deepest
    X_skip = X_skip[::-1]

    # make an encoder model that outputs *all* skip‑tensors
    encoder = Model(inputs=IN,
                    outputs=X_skip,
                    name='swin_unet_encoder')
    return encoder

In [57]:
def make_swin_unet_decoder(patch_size=(4,4),
                           stack_num_up=2,
                           num_heads=[8, 8, 8, 4],  # reversed from encoder
                           window_size=[2, 2, 2, 4],  # adjusted for depth=5
                           num_mlp=1024,
                           shift_window=True,
                           n_labels=1,
                           output_activation='Sigmoid'):

    # 1) define one Input() per skip‑tensor:
    #    deepest latent, then the four skip‑levels above it.
    deepest = Input((16, 1024), name='deepest_in')   # 4×4 patches × dim=1024
    skip3 = Input((64, 512),   name='skip3_in')      # 8×8 patches × dim=512
    skip2 = Input((256, 256),  name='skip2_in')      # 16×16 patches × dim=256
    skip1 = Input((1024, 128), name='skip1_in')      # 32×32 patches × dim=128
    skip0 = Input((4096, 64),  name='skip0_in')      # 64×64 patches × dim=64

    X = deepest
    embed_dim = 1024
    num_patch = (4, 4)

    # --- Decoder Stage -1 (new deepest stage) ---
    X = patch_expanding(num_patch=num_patch,
                        embed_dim=embed_dim,
                        upsample_rate=2,
                        return_vector=True,
                        name='swin_up-1')(X)
    # now: tokens=64, dim=512
    embed_dim //= 2        # → 512
    num_patch = (num_patch[0]*2, num_patch[1]*2)  # → (8,8)

    X = concatenate([X, skip3], axis=-1, name='concat-1')
    X = Dense(embed_dim, use_bias=False, name='proj-1')(X)
    X = swin_transformer_stack(X, stack_num_up,
                               embed_dim, num_patch,
                               num_heads[0], window_size[0],
                               num_mlp, shift_window,
                               name='up-1')

    # --- Decoder Stage 0 ---
    X = patch_expanding(num_patch=num_patch,
                        embed_dim=embed_dim,
                        upsample_rate=2,
                        return_vector=True,
                        name='swin_up0')(X)
    # now: tokens=256, dim=256
    embed_dim //= 2        # → 256
    num_patch = (num_patch[0]*2, num_patch[1]*2)  # → (16,16)

    X = concatenate([X, skip2], axis=-1, name='concat0')
    X = Dense(embed_dim, use_bias=False, name='proj0')(X)
    X = swin_transformer_stack(X, stack_num_up,
                               embed_dim, num_patch,
                               num_heads[1], window_size[1],
                               num_mlp, shift_window,
                               name='up0')

    # --- Decoder Stage 1 ---
    X = patch_expanding(num_patch=num_patch,
                        embed_dim=embed_dim,
                        upsample_rate=2,
                        return_vector=True,
                        name='swin_up1')(X)
    embed_dim //= 2        # → 128
    num_patch = (num_patch[0]*2, num_patch[1]*2)  # → (32,32)

    X = concatenate([X, skip1], axis=-1, name='concat1')
    X = Dense(embed_dim, use_bias=False, name='proj1')(X)
    X = swin_transformer_stack(X, stack_num_up,
                               embed_dim, num_patch,
                               num_heads[2], window_size[2],
                               num_mlp, shift_window,
                               name='up1')

    # --- Decoder Stage 2 ---
    X = patch_expanding(num_patch=num_patch,
                        embed_dim=embed_dim,
                        upsample_rate=2,
                        return_vector=True,
                        name='swin_up2')(X)
    embed_dim //= 2        # → 64
    num_patch = (num_patch[0]*2, num_patch[1]*2)  # → (64,64)

    X = concatenate([X, skip0], axis=-1, name='concat2')
    X = Dense(embed_dim, use_bias=False, name='proj2')(X)
    X = swin_transformer_stack(X, stack_num_up,
                               embed_dim, num_patch,
                               num_heads[3], window_size[3],
                               num_mlp, shift_window,
                               name='up2')

    # --- Final Expanding to Image ---
    X = patch_expanding(num_patch=num_patch,
                        embed_dim=embed_dim,
                        upsample_rate=patch_size[0],
                        return_vector=False,
                        name='swin_up_last')(X)
    # X is now (None, 256, 256, C) – ready for conv
    OUT = CONV_output(X, n_labels,
                      kernel_size=1,
                      activation=output_activation,
                      name='decoder_output')

    decoder = Model(inputs=[deepest, skip3, skip2, skip1, skip0],
                    outputs=OUT,
                    name='swin_unet_decoder')
    return decoder

In [58]:
def Siamese_SwinUnet(input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1)):

    #Define the inputs
    input_1 = Input(shape=input_size1, name='input_1')  # Pre-event optical
    input_2 = Input(shape=input_size2, name='input_2')  # Post-event optical
    # 2. Create Encoder and Decoder
    encoder = make_swin_unet_encoder(input_size= input_size1,
                              filter_num_begin=64,
                              depth=5,
                              stack_num_down=2,
                              patch_size=(4,4),
                              num_heads=[4,8,8,8,8],
                              window_size=[4,2,2,2,2],
                              num_mlp=1024,
                              shift_window=True)
    deep_1, skip3_1, skip2_1, skip1_1, skip0_1 = encoder(input_1)
    deep_2, skip3_2, skip2_2, skip1_2, skip0_2 = encoder(input_2)

    deep_fused = keras.layers.Add()([deep_1, deep_2])

    # Assuming a corresponding decoder architecture
    decoder = make_swin_unet_decoder(patch_size=(4, 4),
                                  stack_num_up=2,
                                  num_heads=[8,8,8,4],       # reversed from encoder
                                  window_size=[2,2,2,4],
                                  num_mlp=1024,         # Reduce MLP size
                                  shift_window=True,
                                  n_labels=1,         # 10 classes for MNIST
                                  output_activation='Sigmoid') # Multi-class classification
    # 3. Connect Encoder and Decoder

    decoder_output_1 = decoder([deep_1, skip3_1, skip2_1, skip1_1, skip0_1])
    decoder_output_2 = decoder([deep_2, skip3_2, skip2_2, skip1_2, skip0_2])
    decoder_output_fused = keras.layers.Multiply()([decoder_output_1, decoder_output_2])

    return Model(inputs=[input_1, input_2], outputs=decoder_output_fused)

In [59]:
# fix random seed for reproducibility
np.random.seed(42)
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
fiilter = 64
learning_rate = 10e-5
# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 16
# Hyperparameters. These are the keys where the associated information for each hyperparameter will be saved.
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []
model_path = f'/content/drive/MyDrive/landslide/Results/weights/Siamese_SwinUnet_size_{size1}_filters_{fiilter}_batch_size_{batch}_lr_{learning_rate}.keras'
# load unet to evaluate the test data
Sia_SwinUnet = Siamese_SwinUnet( input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1))
Sia_SwinUnet.compile(optimizer = Adam(learning_rate = learning_rate), loss = loss, metrics = metrics)
# load the last saved weight from the training
Sia_SwinUnet.load_weights(model_path)

# Get unique ROI values
unique_rois = np.unique(labels)
# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    X_test3_roi = X_test3[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test1_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = Sia_SwinUnet.evaluate([X_test1_roi, X_test2_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("Sia_SwinUnet")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_Sia_SwinUnet_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'patch_expanding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'patch_expanding_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'patch_expanding_2', however the

Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 39s 13s/step - IoU: 0.0037 - accuracy: 0.9482 - f1-score: 0.0074 - loss: 0.9933 - precision: 0.0848 - recall: 0.0039
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - IoU: 0.1364 - accuracy: 0.9026 - f1-score: 0.2401 - loss: 0.6743 - precision: 0.1446 - recall: 0.7054
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - IoU: 0.2365 - accuracy: 0.9778 - f1-score: 0.3825 - loss: 0.5891 - precision: 0.3263 - recall: 0.4621
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - IoU: 0.0416 - accuracy: 0.9589 - f1-score: 0.0798 - loss: 0.8938 - precision: 0.0496 - recall: 0.2048
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 16s 16s/step - IoU: 0.2344 - accuracy: 0.9244 - f1-score: 0.3798 - loss: 0.6401 - precision: 0.4409 - recall: 0.3336


## MODEL TRANSUNET

In [43]:
!pip install keras-unet-collection -q -U

In [44]:
from __future__ import absolute_import

from keras_unet_collection.layer_utils import *
from keras_unet_collection.activations import GELU, Snake
from keras_unet_collection._model_unet_2d import UNET_left, UNET_right
from keras_unet_collection.transformer_layers import patch_extract, patch_embedding
from keras_unet_collection._backbone_zoo import backbone_zoo, bach_norm_checker

import tensorflow as tf
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, MultiHeadAttention, LayerNormalization, Dense, Embedding

In [45]:
# Size of the tiles/patches
size1 = 256 # This line takes the value of the 3rd index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 128.
size2 = 256

# Image bands
img_bands1 = 11 # This line takes the value of the 4th index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 11.

In [46]:
def ViT_MLP(X, filter_num, activation='GELU', name='MLP'):

    activation_func = eval(activation)

    for i, f in enumerate(filter_num):
        X = Dense(f, name='{}_dense_{}'.format(name, i))(X)
        X = activation_func(name='{}_activation_{}'.format(name, i))(X)

    return X

def ViT_block(V, num_heads, key_dim, filter_num_MLP, activation='GELU', name='ViT'):

    # Multiheaded self-attention (MSA)
    V_atten = V # <--- skip
    V_atten = LayerNormalization(name='{}_layer_norm_1'.format(name))(V_atten)
    V_atten = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim,
                                 name='{}_atten'.format(name))(V_atten, V_atten)
    # Skip connection
    V_add = add([V_atten, V], name='{}_skip_1'.format(name)) # <--- skip

    # MLP
    V_MLP = V_add # <--- skip
    V_MLP = LayerNormalization(name='{}_layer_norm_2'.format(name))(V_MLP)
    V_MLP = ViT_MLP(V_MLP, filter_num_MLP, activation, name='{}_mlp'.format(name))
    # Skip connection
    V_out = add([V_MLP, V_add], name='{}_skip_2'.format(name)) # <--- skip

    return V_out


def transunet_2d_base(input_tensor, filter_num, stack_num_down=2, stack_num_up=2,
                      embed_dim=768, num_mlp=3072, num_heads=12, num_transformer=12,
                      activation='ReLU', mlp_activation='GELU', batch_norm=False, pool=True, unpool=True,
                      backbone=None, weights='imagenet', freeze_backbone=True, freeze_batch_norm=True, name='transunet'):

    activation_func = eval(activation)

    X_skip = []
    depth_ = len(filter_num)

    # ----- internal parameters ----- #

    # patch size (fixed to 1-by-1)
    patch_size = 1

    # input tensor size
    input_size = input_tensor.shape[1]

    # encoded feature map size
    encode_size = input_size // 2**(depth_-1)

    # number of size-1 patches
    num_patches = encode_size ** 2

    # dimension of the attention key (= dimension of embedings)
    key_dim = embed_dim

    # number of MLP nodes
    filter_num_MLP = [num_mlp, embed_dim]

    # ----- UNet-like downsampling ----- #

    # no backbone cases
    if backbone is None:

        X = input_tensor

        # stacked conv2d before downsampling
        X = CONV_stack(X, filter_num[0], stack_num=stack_num_down, activation=activation,
                       batch_norm=batch_norm, name='{}_down0'.format(name))
        X_skip.append(X)

        # downsampling blocks
        for i, f in enumerate(filter_num[1:]):
            X = UNET_left(X, f, stack_num=stack_num_down, activation=activation, pool=pool,
                          batch_norm=batch_norm, name='{}_down{}'.format(name, i+1))
            X_skip.append(X)

    # backbone cases
    else:
        # handling VGG16 and VGG19 separately
        if 'VGG' in backbone:
            backbone_ = backbone_zoo(backbone, weights, input_tensor, depth_, freeze_backbone, freeze_batch_norm)
            # collecting backbone feature maps
            X_skip = backbone_([input_tensor,])
            depth_encode = len(X_skip)

        # for other backbones
        else:
            backbone_ = backbone_zoo(backbone, weights, input_tensor, depth_-1, freeze_backbone, freeze_batch_norm)
            # collecting backbone feature maps
            X_skip = backbone_([input_tensor,])
            depth_encode = len(X_skip) + 1


        # extra conv2d blocks are applied
        # if downsampling levels of a backbone < user-specified downsampling levels
        if depth_encode < depth_:

            # begins at the deepest available tensor
            X = X_skip[-1]

            # extra downsamplings
            for i in range(depth_-depth_encode):
                i_real = i + depth_encode

                X = UNET_left(X, filter_num[i_real], stack_num=stack_num_down, activation=activation, pool=pool,
                              batch_norm=batch_norm, name='{}_down{}'.format(name, i_real+1))
                X_skip.append(X)

    # subtrack the last tensor (will be replaced by the ViT output)
    X = X_skip[-1]
    X_skip = X_skip[:-1]

    # 1-by-1 linear transformation before entering ViT blocks
    X = Conv2D(filter_num[-1], 1, padding='valid', use_bias=False, name='{}_conv_trans_before'.format(name))(X)

    X = patch_extract((patch_size, patch_size))(X)
    X = patch_embedding(num_patches, embed_dim)(X)

    # stacked ViTs
    for i in range(num_transformer):
        X = ViT_block(X, num_heads, key_dim, filter_num_MLP, activation=mlp_activation,
                      name='{}_ViT_{}'.format(name, i))

    # reshape patches to feature maps
    X = tf.keras.layers.Reshape((encode_size, encode_size, embed_dim))(X)

    # 1-by-1 linear transformation to adjust the number of channels
    X = Conv2D(filter_num[-1], 1, padding='valid', use_bias=False, name='{}_conv_trans_after'.format(name))(X)

    X_skip.append(X)

    # ----- UNet-like upsampling ----- #

    # reverse indexing encoded feature maps
    X_skip = X_skip[::-1]
    # upsampling begins at the deepest available tensor
    X = X_skip[0]
    # other tensors are preserved for concatenation
    X_decode = X_skip[1:]
    depth_decode = len(X_decode)

    # reverse indexing filter numbers
    filter_num_decode = filter_num[:-1][::-1]

    # upsampling with concatenation
    for i in range(depth_decode):
        X = UNET_right(X, [X_decode[i],], filter_num_decode[i], stack_num=stack_num_up, activation=activation,
                       unpool=unpool, batch_norm=batch_norm, name='{}_up{}'.format(name, i))

    # if tensors for concatenation is not enough
    # then use upsampling without concatenation
    if depth_decode < depth_-1:
        for i in range(depth_-depth_decode-1):
            i_real = i + depth_decode
            X = UNET_right(X, None, filter_num_decode[i_real], stack_num=stack_num_up, activation=activation,
                       unpool=unpool, batch_norm=batch_norm, concat=False, name='{}_up{}'.format(name, i_real))

    return X

def transunet_2d(input_size, filter_num, n_labels, stack_num_down=2, stack_num_up=2,
                 embed_dim=768, num_mlp = 3072, num_heads=12, num_transformer=12,
                 activation='ReLU', mlp_activation='GELU', output_activation='Softmax', batch_norm=False, pool=True, unpool=True,
                 backbone=None, weights='imagenet', freeze_backbone=True, freeze_batch_norm=True, name='transunet'):

    activation_func = eval(activation)

    IN = Input(input_size)

    # base
    X = transunet_2d_base(IN, filter_num, stack_num_down=stack_num_down, stack_num_up=stack_num_up,
                          embed_dim=embed_dim, num_mlp=num_mlp, num_heads=num_heads, num_transformer=num_transformer,
                          activation=activation, mlp_activation=mlp_activation, batch_norm=batch_norm, pool=pool, unpool=unpool,
                          backbone=backbone, weights=weights, freeze_backbone=freeze_backbone, freeze_batch_norm=freeze_batch_norm, name=name)

    # output layer
    OUT = CONV_output(X, n_labels, kernel_size=1, activation=output_activation, name='{}_output'.format(name))

    # functional API model
    model = Model(inputs=IN, outputs=OUT, name='{}_model'.format(name))

    return model

In [49]:
# fix random seed for reproducibility
np.random.seed(42)
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
fiilter = 64
learning_rate = 10e-5

# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 4

# Hyperparameters. These are the keys where the associated information for each hyperparameter will be saved.
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []

model_path = f'/content/drive/MyDrive/landslide/Results/weights/transunet_2d_size_{size1}_filters_{fiilter}_batch_size_{batch}_lr_{learning_rate}.keras'
optimizer = Adam(learning_rate=learning_rate)
Transunet = transunet_2d(
                      (size1,size1,img_bands1),
                      filter_num=[64, 128, 256, 512, 1024],
                      n_labels=1,
                      stack_num_down=2,
                      stack_num_up=2,
                      embed_dim=512,           # Reduced from 1024
                      num_mlp=2048,            # 4 × embed_dim (512 * 4 = 2048)
                      num_heads=8,             # embed_dim // 64 (512 / 64 = 8)
                      num_transformer=6,       # Reduced from 12
                      activation='ReLU',
                      mlp_activation='GELU',
                      output_activation='Sigmoid',
                      batch_norm=True,
                      pool=True,
                      unpool='bilinear',
                      name='transunet'
                  )
Transunet.compile(optimizer = optimizer, loss = loss, metrics = metrics)
# load the last saved weight from the training
Transunet.load_weights(model_path)
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test2_roi = X_test2[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test2_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = Transunet.evaluate(X_test2_roi, y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("Transunet")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_Transunet_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 340 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 64s 17s/step - IoU: 0.1307 - accuracy: 0.9526 - f1-score: 0.2312 - loss: 0.8129 - precision: 0.5966 - recall: 0.1434
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - IoU: 0.2907 - accuracy: 0.9531 - f1-score: 0.4505 - loss: 0.4405 - precision: 0.3025 - recall: 0.8817
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - IoU: 0.2383 - accuracy: 0.9770 - f1-score: 0.3848 - loss: 0.5819 - precision: 0.3196 - recall: 0.4836
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step - IoU: 0.1949 - accuracy: 0.9901 - f1-score: 0.3262 - loss: 0.6982 - precision: 0.3995 - recall: 0.2756
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 38s 38s/step - IoU: 0.4275 - accuracy: 0.9529 - f1-score: 0.5989 - loss: 0.4419 - precision: 0.7323 - recall: 0.5066


## MODEL  SDUNET (TRANSUNET with topo images)

In [50]:
!pip install keras-unet-collection -q -U

In [51]:
from __future__ import absolute_import

from keras_unet_collection.layer_utils import *
from keras_unet_collection.activations import GELU, Snake
from keras_unet_collection._model_unet_2d import UNET_left, UNET_right
from keras_unet_collection.transformer_layers import patch_extract, patch_embedding
from keras_unet_collection._backbone_zoo import backbone_zoo, bach_norm_checker

import tensorflow as tf
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, MultiHeadAttention, LayerNormalization, Dense, Embedding

In [52]:
# Size of the tiles/patches
size1 = 256 # This line takes the value of the 3rd index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 128.
size2 = 256

# Image bands
img_bands1 = 17 # This line takes the value of the 4th index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 11.

In [53]:
def ViT_MLP(X, filter_num, activation='GELU', name='MLP'):

    activation_func = eval(activation)

    for i, f in enumerate(filter_num):
        X = Dense(f, name='{}_dense_{}'.format(name, i))(X)
        X = activation_func(name='{}_activation_{}'.format(name, i))(X)

    return X

def ViT_block(V, num_heads, key_dim, filter_num_MLP, activation='GELU', name='ViT'):

    # Multiheaded self-attention (MSA)
    V_atten = V # <--- skip
    V_atten = LayerNormalization(name='{}_layer_norm_1'.format(name))(V_atten)
    V_atten = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim,
                                 name='{}_atten'.format(name))(V_atten, V_atten)
    # Skip connection
    V_add = add([V_atten, V], name='{}_skip_1'.format(name)) # <--- skip

    # MLP
    V_MLP = V_add # <--- skip
    V_MLP = LayerNormalization(name='{}_layer_norm_2'.format(name))(V_MLP)
    V_MLP = ViT_MLP(V_MLP, filter_num_MLP, activation, name='{}_mlp'.format(name))
    # Skip connection
    V_out = add([V_MLP, V_add], name='{}_skip_2'.format(name)) # <--- skip

    return V_out


def transunet_2d_base(input_tensor, filter_num, stack_num_down=2, stack_num_up=2,
                      embed_dim=768, num_mlp=3072, num_heads=12, num_transformer=12,
                      activation='ReLU', mlp_activation='GELU', batch_norm=False, pool=True, unpool=True,
                      backbone=None, weights='imagenet', freeze_backbone=True, freeze_batch_norm=True, name='transunet'):

    activation_func = eval(activation)

    X_skip = []
    depth_ = len(filter_num)

    # ----- internal parameters ----- #

    # patch size (fixed to 1-by-1)
    patch_size = 1

    # input tensor size
    input_size = input_tensor.shape[1]

    # encoded feature map size
    encode_size = input_size // 2**(depth_-1)

    # number of size-1 patches
    num_patches = encode_size ** 2

    # dimension of the attention key (= dimension of embedings)
    key_dim = embed_dim

    # number of MLP nodes
    filter_num_MLP = [num_mlp, embed_dim]

    # ----- UNet-like downsampling ----- #

    # no backbone cases
    if backbone is None:

        X = input_tensor

        # stacked conv2d before downsampling
        X = CONV_stack(X, filter_num[0], stack_num=stack_num_down, activation=activation,
                       batch_norm=batch_norm, name='{}_down0'.format(name))
        X_skip.append(X)

        # downsampling blocks
        for i, f in enumerate(filter_num[1:]):
            X = UNET_left(X, f, stack_num=stack_num_down, activation=activation, pool=pool,
                          batch_norm=batch_norm, name='{}_down{}'.format(name, i+1))
            X_skip.append(X)

    # backbone cases
    else:
        # handling VGG16 and VGG19 separately
        if 'VGG' in backbone:
            backbone_ = backbone_zoo(backbone, weights, input_tensor, depth_, freeze_backbone, freeze_batch_norm)
            # collecting backbone feature maps
            X_skip = backbone_([input_tensor,])
            depth_encode = len(X_skip)

        # for other backbones
        else:
            backbone_ = backbone_zoo(backbone, weights, input_tensor, depth_-1, freeze_backbone, freeze_batch_norm)
            # collecting backbone feature maps
            X_skip = backbone_([input_tensor,])
            depth_encode = len(X_skip) + 1


        # extra conv2d blocks are applied
        # if downsampling levels of a backbone < user-specified downsampling levels
        if depth_encode < depth_:

            # begins at the deepest available tensor
            X = X_skip[-1]

            # extra downsamplings
            for i in range(depth_-depth_encode):
                i_real = i + depth_encode

                X = UNET_left(X, filter_num[i_real], stack_num=stack_num_down, activation=activation, pool=pool,
                              batch_norm=batch_norm, name='{}_down{}'.format(name, i_real+1))
                X_skip.append(X)

    # subtrack the last tensor (will be replaced by the ViT output)
    X = X_skip[-1]
    X_skip = X_skip[:-1]

    # 1-by-1 linear transformation before entering ViT blocks
    X = Conv2D(filter_num[-1], 1, padding='valid', use_bias=False, name='{}_conv_trans_before'.format(name))(X)

    X = patch_extract((patch_size, patch_size))(X)
    X = patch_embedding(num_patches, embed_dim)(X)

    # stacked ViTs
    for i in range(num_transformer):
        X = ViT_block(X, num_heads, key_dim, filter_num_MLP, activation=mlp_activation,
                      name='{}_ViT_{}'.format(name, i))

    # reshape patches to feature maps
    X = tf.keras.layers.Reshape((encode_size, encode_size, embed_dim))(X)

    # 1-by-1 linear transformation to adjust the number of channels
    X = Conv2D(filter_num[-1], 1, padding='valid', use_bias=False, name='{}_conv_trans_after'.format(name))(X)

    X_skip.append(X)

    # ----- UNet-like upsampling ----- #

    # reverse indexing encoded feature maps
    X_skip = X_skip[::-1]
    # upsampling begins at the deepest available tensor
    X = X_skip[0]
    # other tensors are preserved for concatenation
    X_decode = X_skip[1:]
    depth_decode = len(X_decode)

    # reverse indexing filter numbers
    filter_num_decode = filter_num[:-1][::-1]

    # upsampling with concatenation
    for i in range(depth_decode):
        X = UNET_right(X, [X_decode[i],], filter_num_decode[i], stack_num=stack_num_up, activation=activation,
                       unpool=unpool, batch_norm=batch_norm, name='{}_up{}'.format(name, i))

    # if tensors for concatenation is not enough
    # then use upsampling without concatenation
    if depth_decode < depth_-1:
        for i in range(depth_-depth_decode-1):
            i_real = i + depth_decode
            X = UNET_right(X, None, filter_num_decode[i_real], stack_num=stack_num_up, activation=activation,
                       unpool=unpool, batch_norm=batch_norm, concat=False, name='{}_up{}'.format(name, i_real))

    return X

def transunet_2d(input_size, filter_num, n_labels, stack_num_down=2, stack_num_up=2,
                 embed_dim=768, num_mlp = 3072, num_heads=12, num_transformer=12,
                 activation='ReLU', mlp_activation='GELU', output_activation='Softmax', batch_norm=False, pool=True, unpool=True,
                 backbone=None, weights='imagenet', freeze_backbone=True, freeze_batch_norm=True, name='transunet'):

    activation_func = eval(activation)

    IN = Input(input_size)

    # base
    X = transunet_2d_base(IN, filter_num, stack_num_down=stack_num_down, stack_num_up=stack_num_up,
                          embed_dim=embed_dim, num_mlp=num_mlp, num_heads=num_heads, num_transformer=num_transformer,
                          activation=activation, mlp_activation=mlp_activation, batch_norm=batch_norm, pool=pool, unpool=unpool,
                          backbone=backbone, weights=weights, freeze_backbone=freeze_backbone, freeze_batch_norm=freeze_batch_norm, name=name)

    # output layer
    OUT = CONV_output(X, n_labels, kernel_size=1, activation=output_activation, name='{}_output'.format(name))

    # functional API model
    model = Model(inputs=IN, outputs=OUT, name='{}_model'.format(name))

    return model

In [54]:
# fix random seed for reproducibility
np.random.seed(42)
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
fiilter = 64
learning_rate = 10e-5

# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 4
# Hyperparameters. These are the keys where the associated information for each hyperparameter will be saved.
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []
model_path = f'/content/drive/MyDrive/landslide/Results/weights/transunet2_2d_size_{size1}_filters_{fiilter}_batch_size_{batch}_lr_{learning_rate}.keras'

SDCUnet = transunet_2d(
                      (size1,size1,img_bands1),
                      filter_num=[64, 128, 256, 512, 1024],
                      n_labels=1,
                      stack_num_down=2,
                      stack_num_up=2,
                      embed_dim=512,           # Reduced from 1024
                      num_mlp=2048,            # 4 × embed_dim (512 * 4 = 2048)
                      num_heads=8,             # embed_dim // 64 (512 / 64 = 8)
                      num_transformer=6,       # Reduced from 12
                      activation='ReLU',
                      mlp_activation='GELU',
                      output_activation='Sigmoid',
                      batch_norm=True,
                      pool=True,
                      unpool='bilinear',
                      name='SDCUnet'
                  )
SDCUnet.compile(optimizer = optimizer, loss = loss, metrics = metrics)
# load the last saved weight from the training
SDCUnet.load_weights(model_path)
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test2_roi = X_test2[roi_indices]
    X_test3_roi = X_test3[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test1_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = SDCUnet.evaluate(np.concatenate([X_test2_roi, X_test3_roi], axis=-1), y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("SDCUnet")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_SDCUnet_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 404 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Processing ROI: 0
X_test1_roi shape: (27, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 14s 4s/step - IoU: 0.1891 - accuracy: 0.9521 - f1-score: 0.3168 - loss: 0.7044 - precision: 0.4795 - recall: 0.2662
Processing ROI: 1
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - IoU: 0.2578 - accuracy: 0.9616 - f1-score: 0.4099 - loss: 0.5269 - precision: 0.3081 - recall: 0.6123
Processing ROI: 2
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - IoU: 0.3054 - accuracy: 0.9866 - f1-score: 0.4680 - loss: 0.5659 - precision: 0.5708 - recall: 0.3965
Processing ROI: 3
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - IoU: 0.1327 - accuracy: 0.9906 - f1-score: 0.2343 - loss: 0.7997 - precision: 0.4003 - recall: 0.1656
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - IoU: 0.4180 - accuracy: 0.9519 - f1-score: 0.5895 - loss: 0.4510 - precision: 0.7231 - recall: 0.4976


## MODEL BBUNET

In [55]:
size1 = 256
img_bands1= 11
size2 = 256
img_bands2 = 6

In [56]:
import os
import numpy as np
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers, ops
from tensorflow.keras.layers import Conv2D, BatchNormalization, ReLU, Add, Subtract, Concatenate, Dropout, MaxPooling2D, Conv2DTranspose, ReLU, Concatenate, Activation, Input
from tensorflow.keras.models import Model

In [57]:
# Common building blocks
def conv_block(x, filters, block_num, conv_num):
    x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal', name=f'conv{block_num}{conv_num}')(x)
    x = BatchNormalization(name=f'bn{block_num}{conv_num}')(x)
    x = ReLU()(x)
    x = Dropout(0.2, name=f'do{block_num}{conv_num}')(x)
    return x

def fusion_module(F_pre, F_post, F_dem, filters, name='fusion_module'):
    """
    F_pre, F_post, F_dem: input tensors of shape (H, W, C)
    filters: number of output filters C
    """
    with tf.name_scope(name):
        # Step 1: Difference between S2-pre and S2-post
        diff = Subtract()([F_pre, F_post])  # shape: (H, W, C)

        # Step 2: Concatenate with DEM
        concat = Concatenate(axis=-1)([diff, F_dem])  # shape: (H, W, 2C)

        # Step 3: Conv-BN-ReLU -> Conv-BN
        x = Conv2D(filters, kernel_size=3, padding='same', kernel_initializer='he_normal')(concat)
        x = BatchNormalization()(x)
        x = Conv2D(filters, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = ReLU()(x)

        # Step 4: Add result with F_post
        fused = Add()([x, F_post])  # shape: (H, W, C)

        return fused

In [58]:
def Main_branch_encoder(filtersFirstLayer, input_shape):
    # Define input within the function
    input_tensor = Input(shape=input_shape)

    # Stage 1
    x11 = conv_block(input_tensor, filtersFirstLayer, 1, 1)
    x12 = conv_block(x11, filtersFirstLayer, 1, 2)
    x1p = MaxPooling2D(pool_size=2, strides=2)(x12)

    # Stage 2
    x21 = conv_block(x1p, filtersFirstLayer*2, 2, 1)
    x22 = conv_block(x21, filtersFirstLayer*2, 2, 2)
    x2p = MaxPooling2D(pool_size=2, strides=2)(x22)

    # Stage 3
    x31 = conv_block(x2p, filtersFirstLayer*4, 3, 1)
    x32 = conv_block(x31, filtersFirstLayer*4, 3, 2)
    x33 = conv_block(x32, filtersFirstLayer*4, 3, 3)
    x3p = MaxPooling2D(pool_size=2, strides=2)(x33)

    # Stage 4
    x41 = conv_block(x3p, filtersFirstLayer*8, 4, 1)
    x42 = conv_block(x41, filtersFirstLayer*8, 4, 2)
    x43 = conv_block(x42, filtersFirstLayer*8, 4, 3)
    x4p = MaxPooling2D(pool_size=2, strides=2)(x43)

    # Stage 5
    x51 = conv_block(x4p, filtersFirstLayer*16, 5, 1)
    x52 = conv_block(x51, filtersFirstLayer*16, 5, 2)
    x53 = conv_block(x52, filtersFirstLayer*16, 5, 3)
    x5p = MaxPooling2D(pool_size=2, strides=2)(x53)
    return Model(inputs=input_tensor, outputs= [x5p, x4p, x3p, x2p, x1p])

In [59]:
def BBUnet(filtersFirstLayer, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2)):

    #Define the inputs
    input_1 = Input(shape=input_size1, name='input_1')  # Pre-event optical
    input_2 = Input(shape=input_size2, name='input_2')  # Post-event optical
    input_3 = Input(shape=input_size3, name='input_3')  # Topographic

    # Define shared encoder with correct input shape
    encoder_model = Main_branch_encoder(filtersFirstLayer, input_shape = input_size1)

    # Apply encoder to both concatenated inputs
    Features1 = encoder_model(input_1)
    Features2 = encoder_model(input_2)

    # Unpacking encoder outputs
    F5_1, F4_1, F3_1, F2_1, F1_1 = Features1
    F5_2, F4_2, F3_2, F2_2, F1_2 = Features2

    # Processing for second input (same architecture)
    # Stage 1
    y11 = conv_block(input_3, filtersFirstLayer, 1, 1)
    y12 = conv_block(y11, filtersFirstLayer, 1, 2)
    y1p = MaxPooling2D(pool_size=2, strides=2)(y12)

    # Fusion Stage 1
    FF_1 = fusion_module(F1_1, F1_2, y1p, filtersFirstLayer)

    # Stage 2
    y21 = conv_block(y1p, filtersFirstLayer*2, 2, 1)
    y22 = conv_block(y21, filtersFirstLayer*2, 2, 2)
    y2p = MaxPooling2D(pool_size=2, strides=2)(y22)

    # Fusion Stage 2
    FF_2 = fusion_module(F2_1, F2_2, y2p, filtersFirstLayer*2)

    # Stage 3
    y31 = conv_block(y2p, filtersFirstLayer*4, 3, 1)
    y32 = conv_block(y31, filtersFirstLayer*4, 3, 2)
    y3p = MaxPooling2D(pool_size=2, strides=2)(y32)

    # Fusion Stage 3
    FF_3 = fusion_module(F3_1, F3_2, y3p, filtersFirstLayer*4)

    # Stage 4
    y41 = conv_block(y3p, filtersFirstLayer*8, 4, 1)
    y42 = conv_block(y41, filtersFirstLayer*8, 4, 2)
    y4p = MaxPooling2D(pool_size=2, strides=2)(y42)

    # Fusion Stage 4
    FF_4 = fusion_module(F4_1, F4_2, y4p, filtersFirstLayer*8)

    # Stage 5
    y51 = conv_block(y4p, filtersFirstLayer*16, 5, 1)
    y52 = conv_block(y51, filtersFirstLayer*16, 5, 2)
    y5p = MaxPooling2D(pool_size=2, strides=2)(y52)

    # Fusion Stage 5
    FF_5 = fusion_module(F5_1, F5_2, y5p, filtersFirstLayer*16)

    # Decoder path

    # Stage 5d
    x5d = Conv2DTranspose(filtersFirstLayer*8, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv5')(FF_5)

    # Concatenate with absolute difference
    x5d_concat = Concatenate()([x5d, FF_4])

    x52d = conv_block(x5d_concat, filtersFirstLayer*8, 'd5', 2)
    x51d = conv_block(x52d, filtersFirstLayer*8, 'd5', 1)

    # Stage 4d
    x4d = Conv2DTranspose(filtersFirstLayer*4, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv4')(x51d)

    # Concatenate with absolute difference
    x4d_concat = Concatenate()([x4d, FF_3])

    x42d = conv_block(x4d_concat, filtersFirstLayer*4, 'd4', 2)
    x41d = conv_block(x42d, filtersFirstLayer*4, 'd4', 1)

    # Stage 3d
    x3d = Conv2DTranspose(filtersFirstLayer*2, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv3')(x41d)

    # Concatenate with absolute difference
    x3d_concat = Concatenate()([x3d, FF_2])

    x32d = conv_block(x3d_concat, filtersFirstLayer*2, 'd3', 2)
    x31d = conv_block(x32d, filtersFirstLayer*2, 'd3', 1)

    # Stage 2d
    x2d = Conv2DTranspose(filtersFirstLayer, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv2')(x31d)

    # Concatenate with absolute difference
    x2d_concat = Concatenate()([x2d, FF_1])

    x22d = conv_block(x2d_concat, filtersFirstLayer, 'd2', 2)
    x21d = conv_block(x22d, filtersFirstLayer, 'd2', 1)

    #Stage 1d
    x1d = Conv2DTranspose(filtersFirstLayer, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv1')(x21d)
    Final_output = Conv2D(1, 1, padding = 'same', kernel_initializer='he_normal') (x1d)
    Final_output = Activation('sigmoid')(Final_output)

    return Model(inputs=[input_1, input_2, input_3], outputs=Final_output)

In [60]:
# fix random seed for reproducibility
np.random.seed(42)
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
fiilter = 64
learning_rate = 10e-5

# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 4
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []
model_path = f'/content/drive/MyDrive/landslide/Results/weights/BBUnet2_size_{size1}_filters_{fiilter}_batch_size_{batch}_lr_{learning_rate}.keras'

# load unet to evaluate the test data
BBUNet = BBUnet(filtersFirstLayer=fiilter, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2))
BBUNet.compile(optimizer = Adam(learning_rate = learning_rate), loss = loss, metrics = metrics)
# load the last saved weight from the training
BBUNet.load_weights(model_path)
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    X_test3_roi = X_test3[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test1_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = BBUNet.evaluate([X_test1_roi, X_test2_roi, X_test3_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("BBUNet")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_BBUNet_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 354 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 59s 18s/step - IoU: 0.2638 - accuracy: 0.9554 - f1-score: 0.4174 - loss: 0.6293 - precision: 0.5760 - recall: 0.3304
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - IoU: 0.2603 - accuracy: 0.9435 - f1-score: 0.4131 - loss: 0.4709 - precision: 0.2670 - recall: 0.9118
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - IoU: 0.4645 - accuracy: 0.9903 - f1-score: 0.6343 - loss: 0.3963 - precision: 0.7272 - recall: 0.5625
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 17s 17s/step - IoU: 0.3224 - accuracy: 0.9920 - f1-score: 0.4876 - loss: 0.5334 - precision: 0.5495 - recall: 0.4383
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 36s 36s/step - IoU: 0.2527 - accuracy: 0.9408 - f1-score: 0.4034 - loss: 0.6519 - precision: 0.6692 - recall: 0.2887


## MODEL SIAUNET

In [61]:
size1 = 256
img_bands1= 11
size2 = 256
img_bands2 = 6

In [62]:
import os
import numpy as np
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers, ops
from tensorflow.keras.layers import Conv2D, BatchNormalization, ReLU, Add, Subtract, Concatenate, Dropout, MaxPooling2D, Conv2DTranspose, ReLU, Concatenate, Activation, Input
from tensorflow.keras import backend as K
from tensorflow.keras.layers import AveragePooling2D, Lambda, Conv2D, Conv2DTranspose, Activation, Reshape, concatenate, Concatenate, BatchNormalization, ZeroPadding2D, UpSampling2D
from tensorflow.keras.models import Model
from tensorflow.keras import layers, Sequential

In [63]:
@tf.keras.utils.register_keras_serializable()
class ChannelAttentionModule(tf.keras.layers.Layer):
    def __init__(self, in_channels, reduction_ratio=16):
        super(ChannelAttentionModule, self).__init__()

        self.in_channels = in_channels
        self.reduction_ratio = reduction_ratio

        self.avgpool1 = layers.GlobalAveragePooling2D()
        self.maxpool1 = layers.GlobalMaxPool2D()

        # Shared MLP
        self.dense1 = layers.Dense(in_channels // reduction_ratio, activation='relu')
        self.dense2 = layers.Dense(in_channels, activation='relu')

        self.ac_channel = layers.Activation('sigmoid')
        self.reshape_channel = layers.Reshape((1, 1, in_channels))

    def call(self, x):

        # Channel Attention
        avgpool = self.avgpool1(x)
        maxpool = self.maxpool1(x)
        avg_out = self.dense2(self.dense1(avgpool))
        max_out = self.dense2(self.dense1(maxpool))
        channel_out = layers.add([avg_out, max_out])
        channel_out = self.ac_channel(channel_out)
        channel_out = self.reshape_channel(channel_out)

        return channel_out

    def get_config(self):
        base_config = super(ChannelAttentionModule, self).get_config()
        base_config['in_channels'] = self.in_channels
        base_config['reduction_ratio'] = self.reduction_ratio
        return base_config

@tf.keras.utils.register_keras_serializable()
class SpatialAttentionModule(tf.keras.layers.Layer):
    def __init__(self):
        super(SpatialAttentionModule, self).__init__()

        self.conv_spatial = layers.Conv2D(1, 7, padding='same')
        self.ac_spatial = layers.Activation('sigmoid')

    def call(self, x):

        # Spatial Attention
        avgpool = tf.reduce_mean(x, axis=3, keepdims=True)
        maxpool = tf.reduce_max(x, axis=3, keepdims=True)
        spatial = layers.Concatenate(axis=3)([avgpool, maxpool])
        spatial = self.conv_spatial(spatial)
        spatial_out = self.ac_spatial(spatial)

        return spatial_out

    def get_config(self):
        base_config = super(SpatialAttentionModule, self).get_config()
        return base_config

@tf.keras.utils.register_keras_serializable()
class CBAM(tf.keras.layers.Layer):
    def __init__(self, in_channels, reduction_ratio=16):
        super(CBAM, self).__init__()
        self.in_channels = in_channels
        self.reduction_ratio = reduction_ratio

        self.cam = ChannelAttentionModule(in_channels, reduction_ratio)
        self.sam = SpatialAttentionModule()

    def call(self, x):

        x = self.cam(x) * x
        x = self.sam(x) * x

        return x

    def get_config(self):
        base_config = super(CBAM, self).get_config()
        base_config['in_channels'] = self.in_channels
        base_config['reduction_ratio'] = self.reduction_ratio
        return base_config

In [64]:
# Common building blocks
def conv_block(x, filters, block_num, conv_num):
    x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal', name=f'conv{block_num}{conv_num}')(x)
    x = BatchNormalization(name=f'bn{block_num}{conv_num}')(x)
    x = ReLU()(x)
    x = Dropout(0.2, name=f'do{block_num}{conv_num}')(x)
    return x

def Upsample(tensor, size):  # Remove interpolation
    '''bilinear upsampling'''
    name = tensor.name.split('/')[0] + '_upsample'

    # Replace tf.image.resize and Lambda with UpSampling2D
    y = UpSampling2D(size=size, interpolation='bilinear', name=name)(tensor)
    return y

In [65]:
def ASPP(tensor, filtersFirstLayer):
    '''atrous spatial pyramid pooling'''
    dims = K.int_shape(tensor)

    y_pool = AveragePooling2D(pool_size=(
        dims[1], dims[2]), name='average_pooling')(tensor)
    y_pool = Conv2D(filters=filtersFirstLayer, kernel_size=1, padding='same',
                    kernel_initializer='he_normal', name='pool_1x1conv2d', use_bias=False)(y_pool)
    y_pool = BatchNormalization(name=f'bn_1')(y_pool)
    y_pool = Activation('relu', name=f'relu_1')(y_pool)

    y_pool = Upsample(tensor=y_pool, size=[dims[1], dims[2]])

    y_1 = Conv2D(filters=filtersFirstLayer, kernel_size=1, dilation_rate=1, padding='same',
                 kernel_initializer='he_normal', name='ASPP_conv2d_d1', use_bias=False)(tensor)
    y_1 = BatchNormalization(name=f'bn_2')(y_1)
    y_1 = Activation('relu', name=f'relu_2')(y_1)

    y_6 = Conv2D(filters=filtersFirstLayer, kernel_size=3, dilation_rate=6, padding='same',
                 kernel_initializer='he_normal', name='ASPP_conv2d_d6', use_bias=False)(tensor)
    y_6 = BatchNormalization(name=f'bn_3')(y_6)
    y_6 = Activation('relu', name=f'relu_3')(y_6)

    y_12 = Conv2D(filters=filtersFirstLayer, kernel_size=3, dilation_rate=12, padding='same',
                  kernel_initializer='he_normal', name='ASPP_conv2d_d12', use_bias=False)(tensor)
    y_12 = BatchNormalization(name=f'bn_4')(y_12)
    y_12 = Activation('relu', name=f'relu_4')(y_12)

    y_18 = Conv2D(filters=filtersFirstLayer, kernel_size=3, dilation_rate=18, padding='same',
                  kernel_initializer='he_normal', name='ASPP_conv2d_d18', use_bias=False)(tensor)
    y_18 = BatchNormalization(name=f'bn_5')(y_18)
    y_18 = Activation('relu', name=f'relu_5')(y_18)

    y = concatenate([y_pool, y_1, y_6, y_12, y_18], name='ASPP_concat')

    y = Conv2D(filters=filtersFirstLayer, kernel_size=1, dilation_rate=1, padding='same',
               kernel_initializer='he_normal', name='ASPP_conv2d_final', use_bias=False)(y)
    y = BatchNormalization(name=f'bn_final')(y)
    y = Activation('relu', name=f'relu_final')(y)
    return y

In [66]:
def Main_branch_encoder(filtersFirstLayer, input_shape):
    # Define input within the function
    input_tensor = Input(shape=input_shape)

    # Stage 1
    x11 = conv_block(input_tensor, filtersFirstLayer, 1, 1)
    x12 = conv_block(x11, filtersFirstLayer, 1, 2)
    #x12 = DualAttentionModule(in_channels=filtersFirstLayer)(x12)
    x12 = CBAM(in_channels=filtersFirstLayer)(x12)
    x1p = MaxPooling2D(pool_size=2, strides=2)(x12)

    # Stage 2
    x21 = conv_block(x1p, filtersFirstLayer*2, 2, 1)
    x22 = conv_block(x21, filtersFirstLayer*2, 2, 2)
    #x22 = DualAttentionModule(in_channels=filtersFirstLayer*2)(x22)
    x22 = CBAM(in_channels=filtersFirstLayer*2)(x22)
    x2p = MaxPooling2D(pool_size=2, strides=2)(x22)

    # Stage 3
    x31 = conv_block(x2p, filtersFirstLayer*4, 3, 1)
    x32 = conv_block(x31, filtersFirstLayer*4, 3, 2)
    #x32 = DualAttentionModule(in_channels=filtersFirstLayer*4)(x32)
    x32 = CBAM(in_channels=filtersFirstLayer*4)(x32)
    x3p = MaxPooling2D(pool_size=2, strides=2)(x32)

    # Stage 4
    x41 = conv_block(x3p, filtersFirstLayer*8, 4, 1)
    x42 = conv_block(x41, filtersFirstLayer*8, 4, 2)
    #x42 = DualAttentionModule(in_channels=filtersFirstLayer*8)(x42)
    x42 = CBAM(in_channels=filtersFirstLayer*8)(x42)
    x4p = MaxPooling2D(pool_size=2, strides=2)(x42)

    # Stage 5
    x51 = conv_block(x4p, filtersFirstLayer*16, 5, 1)
    x52 = conv_block(x51, filtersFirstLayer*16, 5, 2)
    x52 = ASPP(x52, filtersFirstLayer=filtersFirstLayer*16)
    return Model(inputs=input_tensor, outputs= [x52, x42, x32, x22, x12])

In [67]:
def SAUnet(filtersFirstLayer, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1)):

    #Define the inputs
    input_1 = Input(shape=input_size1, name='input_1')  # Pre-event optical
    input_2 = Input(shape=input_size2, name='input_2')  # Post-event optical

    # Define shared encoder with correct input shape
    encoder_model = Main_branch_encoder(filtersFirstLayer, input_shape = input_size1)

    # Apply encoder to both concatenated inputs
    Features1 = encoder_model(input_1)
    Features2 = encoder_model(input_2)

    # Unpacking encoder outputs
    F5_1, F4_1, F3_1, F2_1, F1_1 = Features1
    F5_2, F4_2, F3_2, F2_2, F1_2 = Features2

    # Concatenate with absolute difference
    x5d_concat = Concatenate()([F5_1, F5_2])
    x4d_concat = Concatenate()([F4_1, F4_2])
    x3d_concat = Concatenate()([F3_1, F3_2])
    x2d_concat = Concatenate()([F2_1, F2_2])
    x1d_concat = Concatenate()([F1_1, F1_2])

    # Stage 5d
    x5d = Conv2DTranspose(filtersFirstLayer*16, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv5')(x5d_concat)
    merge5 = concatenate([x5d, x4d_concat])
    x51d = conv_block(merge5, filtersFirstLayer*16, 'd5', 1)
    x52d = conv_block(x51d, filtersFirstLayer*16, 'd5', 2)

    # Stage 4d
    x4d = Conv2DTranspose(filtersFirstLayer*8, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv4')(x52d)
    merge4 = concatenate([x4d, x3d_concat])
    x41d = conv_block(merge4, filtersFirstLayer*8, 'd4', 1)
    x42d = conv_block(x41d, filtersFirstLayer*8, 'd4', 2)

    # Stage 3d
    x3d = Conv2DTranspose(filtersFirstLayer*4, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv3')(x42d)
    merge3 = concatenate([x3d, x2d_concat])
    x31d = conv_block(merge3, filtersFirstLayer*4, 'd3', 1)
    x32d = conv_block(x31d, filtersFirstLayer*4, 'd3', 2)

    # Stage 2d
    x2d = Conv2DTranspose(filtersFirstLayer*2, 3, padding='same', strides=2, kernel_initializer='he_normal', name='upconv2')(x32d)
    merge2 = concatenate([x2d, x1d_concat])
    x21d = conv_block(merge2, filtersFirstLayer*2, 'd2', 1)
    x22d = conv_block(x21d, filtersFirstLayer*2, 'd2', 2)

    #Stage final
    x11d = conv_block(x22d, filtersFirstLayer, 'd1', 1)
    Final_output = Conv2D(1, 1, padding = 'same', kernel_initializer='he_normal') (x11d)
    Final_output = Activation('sigmoid')(Final_output)

    return Model(inputs=[input_1, input_2], outputs=Final_output)

In [68]:
# fix random seed for reproducibility
np.random.seed(42)

metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
fiilter = 64
learning_rate = 10e-5

# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 4
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []
model_path = f'/content/drive/MyDrive/landslide/Results/weights/SAUnet_size_{size1}_filters_{fiilter}_batch_size_{batch}_lr_{learning_rate}.keras'

# load unet to evaluate the test data
SiAUnet = SAUnet(filtersFirstLayer=fiilter, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1))
SiAUnet.compile(optimizer = Adam(learning_rate = learning_rate), loss = loss, metrics = metrics)
# load the last saved weight from the training
SiAUnet.load_weights(model_path)
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test1_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = SiAUnet.evaluate([X_test1_roi, X_test2_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("SiAUnet")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_SiAUnet_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 258 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 158s 43s/step - IoU: 0.2726 - accuracy: 0.9506 - f1-score: 0.4279 - loss: 0.5808 - precision: 0.4825 - recall: 0.3850
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 16s 16s/step - IoU: 0.2550 - accuracy: 0.9379 - f1-score: 0.4064 - loss: 0.4702 - precision: 0.2567 - recall: 0.9748
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - IoU: 0.4096 - accuracy: 0.9822 - f1-score: 0.5812 - loss: 0.3403 - precision: 0.4468 - recall: 0.8309
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 27s 27s/step - IoU: 0.2276 - accuracy: 0.9846 - f1-score: 0.3708 - loss: 0.5801 - precision: 0.2876 - recall: 0.5219
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 103s 103s/step - IoU: 0.4611 - accuracy: 0.9445 - f1-score: 0.6312 - loss: 0.3491 - precision: 0.5857 - recall: 0.6844


## MODEL SegFormer V2

In [69]:
!pip install keras-unet-collection -q -U

In [70]:
from tensorflow import keras
from tensorflow.keras import layers, Model
import tensorflow as tf
!git clone https://github.com/IMvision12/SegFormer-tf
!cd SegFormer-tf
!cp SegFormer-tf/models/* copied_models/
import keras
from keras import ops
import math
from tensorflow.keras.activations import sigmoid

Cloning into 'SegFormer-tf'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 175 (delta 18), reused 18 (delta 11), pack-reused 133 (from 1)
Receiving objects: 100% (175/175), 6.21 MiB | 9.64 MiB/s, done.
Resolving deltas: 100% (71/71), done.


In [71]:
# Size of the tiles/patches
size1 = 256 # This line takes the value of the 3rd index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 128.
size2 = 256
# Image bands
img_bands1 = 11 # This line takes the value of the 4th index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 11.

In [72]:
class Attention(keras.layers.Layer):
    def __init__(
        self,
        dim,
        num_heads,
        sr_ratio,
        qkv_bias=False,
        attn_drop=0.0,
        proj_drop=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = self.dim // self.num_heads

        self.units = self.num_heads * self.head_dim
        self.sqrt_of_units = math.sqrt(self.head_dim)

        self.q = keras.layers.Dense(self.units)
        self.k = keras.layers.Dense(self.units)
        self.v = keras.layers.Dense(self.units)

        self.attn_drop = keras.layers.Dropout(attn_drop)

        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = keras.layers.Conv2D(
                filters=dim, kernel_size=sr_ratio, strides=sr_ratio, name='sr',
            )
            self.norm = keras.layers.LayerNormalization(epsilon=1e-05)

        self.proj = keras.layers.Dense(dim)
        self.proj_drop = keras.layers.Dropout(proj_drop)

    def call(self, x, H, W):
        get_shape = ops.shape(x)
        B = get_shape[0]
        C = get_shape[2]

        q = self.q(x)
        q = ops.reshape(q, (B, -1, self.num_heads, self.head_dim))
        q = ops.transpose(q, axes=[0, 2, 1, 3])  # Shape: [B, nh, T_q, hd]

        if self.sr_ratio > 1:
            x = ops.reshape(x, (B, H, W, C))
            x = self.sr(x)
            x = ops.reshape(x, (B, -1, C))
            x = self.norm(x)

        k = self.k(x)
        k = ops.reshape(k, (B, -1, self.num_heads, self.head_dim))
        k = ops.transpose(k, axes=[0, 2, 1, 3])  # Shape: [B, nh, T_k, hd]

        v = self.v(x)
        v = ops.reshape(v, (B, -1, self.num_heads, self.head_dim))
        v = ops.transpose(v, axes=[0, 2, 1, 3])  # Shape: [B, nh, T_v, hd]

        # Transpose k to [B, nh, hd, T_k] for correct matrix multiplication
        k_transposed = ops.transpose(k, axes=[0, 1, 3, 2])

        # Compute attention scores: [B, nh, T_q, T_k]
        attn = ops.matmul(q, k_transposed)
        scale = ops.cast(self.sqrt_of_units, dtype=attn.dtype)
        attn = attn / scale

        attn = ops.softmax(attn, axis=-1)
        attn = self.attn_drop(attn)

        # Multiply with value vectors
        x = ops.matmul(attn, v)  # Shape: [B, nh, T_q, hd]
        x = ops.transpose(x, axes=[0, 2, 1, 3])
        x = ops.reshape(x, (B, -1, self.units))
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [73]:
from copied_models.utils import DropPath

class DWConv(keras.layers.Layer):
    def __init__(self, filters=768, **kwargs):
        super().__init__(**kwargs)
        self.dwconv = keras.layers.Conv2D(
            filters=filters,
            kernel_size=3,
            strides=1,
            padding="same",
            groups=filters,
        )

    def call(self, x, H, W):
        get_shape_1 = ops.shape(x)
        x = ops.reshape(x, (get_shape_1[0], H, W, get_shape_1[-1]))
        x = self.dwconv(x)
        get_shape_2 = ops.shape(x)
        x = ops.reshape(
            x, (get_shape_2[0], get_shape_2[1] * get_shape_2[2], get_shape_2[3])
        )
        return x


class Mlp(keras.layers.Layer):
    def __init__(
        self,
        in_features,
        hidden_features=None,
        out_features=None,
        drop=0.0,
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = keras.layers.Dense(hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = keras.layers.Activation("gelu")
        self.fc2 = keras.layers.Dense(out_features)
        self.drop = keras.layers.Dropout(drop)

    def call(self, x, H, W):
        x = self.fc1(x)
        x = self.dwconv(x, H=H, W=W)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Block(keras.layers.Layer):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        qkv_bias=False,
        drop=0.0,
        attn_drop=0.0,
        drop_path=0.0,
        sr_ratio=1,
    ):
        super().__init__()
        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-05)
        self.attn = Attention(
            dim,
            num_heads,
            sr_ratio,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.drop_path = DropPath(drop_path)
        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-05)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            drop=drop,
        )

    def call(self, x, H, W):
        # Apply LayerNormalization and Attention layer
        attn_output_norm = self.norm1(x)
        attn_output = self.attn(attn_output_norm, H=H, W=W)
        attn_output_with_drop = self.drop_path(attn_output)
        x = x + attn_output_with_drop

        # Apply LayerNormalization and MLP layer
        mlp_output_norm = self.norm2(x)
        mlp_output = self.mlp(mlp_output_norm, H=H, W=W)
        mlp_output_with_drop = self.drop_path(mlp_output)
        x = x + mlp_output_with_drop

        return x



class OverlapPatchEmbed(keras.layers.Layer):
    def __init__(
        self, img_size=224, patch_size=7, stride=4, filters=768, **kwargs
    ):
        super().__init__(**kwargs)
        self.pad = keras.layers.ZeroPadding2D(padding=patch_size // 2)
        self.conv = keras.layers.Conv2D(
            filters=filters,
            kernel_size=patch_size,
            strides=stride,
            padding="VALID",
            name='proj',
        )
        self.norm = keras.layers.LayerNormalization(epsilon=1e-05)

    def call(self, x):
        x = self.conv(self.pad(x))
        get_shapes = ops.shape(x)
        H = get_shapes[1]
        W = get_shapes[2]
        C = get_shapes[3]
        x = ops.reshape(x, (-1, H * W, C))
        x = self.norm(x)
        return x, H, W


class MixVisionTransformer(keras.layers.Layer):
    def __init__(
        self,
        img_size=256,
        embed_dims=[64, 128, 256, 512, 1024],  # Added 1024
        num_heads=[1, 2, 4, 8, 16],            # Added 16
        mlp_ratios=[4, 4, 4, 4, 4],             # Extended
        qkv_bias=False,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
        depths=[3, 4, 6, 3, 3],                # Added 3
        sr_ratios=[8, 4, 2, 1, 1],             # Added 1
    ):
        super().__init__()
        self.depths = depths

        # patch_embed
        self.patch_embed1 = OverlapPatchEmbed(
            img_size=img_size,
            patch_size=7,
            stride=4,
            filters=embed_dims[0],
        )
        self.patch_embed2 = OverlapPatchEmbed(
            img_size=img_size // 4,
            patch_size=3,
            stride=2,
            filters=embed_dims[1],
        )
        self.patch_embed3 = OverlapPatchEmbed(
            img_size=img_size // 8,
            patch_size=3,
            stride=2,
            filters=embed_dims[2],
        )
        self.patch_embed4 = OverlapPatchEmbed(
            img_size=img_size // 16,
            patch_size=3,
            stride=2,
            filters=embed_dims[3],
        )
        # New patch embed for 5th stage
        self.patch_embed5 = OverlapPatchEmbed(
            img_size=img_size // 32,
            patch_size=3,
            stride=2,
            filters=embed_dims[4],
        )

        # Drop path rates
        dpr = [x for x in ops.linspace(0.0, drop_path_rate, sum(depths))]
        cur = 0

        # Block 1
        self.block1 = [
            Block(
                dim=embed_dims[0],
                num_heads=num_heads[0],
                mlp_ratio=mlp_ratios[0],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[0],
            )
            for i in range(depths[0])
        ]
        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-05)
        cur += depths[0]

        # Block 2
        self.block2 = [
            Block(
                dim=embed_dims[1],
                num_heads=num_heads[1],
                mlp_ratio=mlp_ratios[1],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[1],
            )
            for i in range(depths[1])
        ]
        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-05)
        cur += depths[1]

        # Block 3
        self.block3 = [
            Block(
                dim=embed_dims[2],
                num_heads=num_heads[2],
                mlp_ratio=mlp_ratios[2],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[2],
            )
            for i in range(depths[2])
        ]
        self.norm3 = keras.layers.LayerNormalization(epsilon=1e-05)
        cur += depths[2]

        # Block 4
        self.block4 = [
            Block(
                dim=embed_dims[3],
                num_heads=num_heads[3],
                mlp_ratio=mlp_ratios[3],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[3],
            )
            for i in range(depths[3])
        ]
        self.norm4 = keras.layers.LayerNormalization(epsilon=1e-05)
        cur += depths[3]

        # New Block 5
        self.block5 = [
            Block(
                dim=embed_dims[4],
                num_heads=num_heads[4],
                mlp_ratio=mlp_ratios[4],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[4],
            )
            for i in range(depths[4])
        ]
        self.norm5 = keras.layers.LayerNormalization(epsilon=1e-05)

    def call_features(self, x):
        B = ops.shape(x)[0]
        outs = []

        # stage 1
        x, H, W = self.patch_embed1(x)
        for i, blk in enumerate(self.block1):
            x = blk(x, H=H, W=W)
        x = self.norm1(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # stage 2
        x, H, W = self.patch_embed2(x)
        for i, blk in enumerate(self.block2):
            x = blk(x, H=H, W=W)
        x = self.norm2(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # stage 3
        x, H, W = self.patch_embed3(x)
        for i, blk in enumerate(self.block3):
            x = blk(x, H=H, W=W)
        x = self.norm3(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # stage 4
        x, H, W = self.patch_embed4(x)
        for i, blk in enumerate(self.block4):
            x = blk(x, H=H, W=W)
        x = self.norm4(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # New stage 5
        x, H, W = self.patch_embed5(x)
        for i, blk in enumerate(self.block5):
            x = blk(x, H=H, W=W)
        x = self.norm5(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        return outs

    def call(self, x):
        x = self.call_features(x)
        return x

In [74]:
from copied_models.Head import SegFormerHead
from copied_models.utils import ResizeLayer

MODEL_CONFIGS = {
    "mit_b3": {
        "embed_dims": [64, 128, 320, 512, 1024],
        "depths": [3, 3, 4, 6, 3],
        "decode_dim": 768,
    }}



def SegFormer_B3(input_shape, num_classes):
    input_layer = keras.layers.Input(shape=input_shape)
    x = MixVisionTransformer(
        img_size=input_shape[1],
        embed_dims=MODEL_CONFIGS["mit_b3"]["embed_dims"],
        depths=MODEL_CONFIGS["mit_b3"]["depths"],
    )(input_layer)
    x = SegFormerHead(
        num_mlp_layers=5, num_classes=num_classes,
        decode_dim=MODEL_CONFIGS["mit_b3"]["decode_dim"],
    )(x)

    x = ResizeLayer(input_shape[0], input_shape[1])(x)
    x = sigmoid(x)
    return keras.Model(inputs=input_layer, outputs=x)

In [75]:
# fix random seed for reproducibility
np.random.seed(42)
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
fiilter = 64
learning_rate = 10e-5
# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 4
dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []

model_path = f'/content/drive/MyDrive/landslide/Results/weights/SegFormer_B3_size_{size1}_filters_{fiilter}_batch_size_{batch}_lr_{learning_rate}.keras'
# load unet to evaluate the test data
SegFormer = SegFormer_B3(input_shape = (size1,size1,img_bands1), num_classes =1)
SegFormer.compile(optimizer = Adam(learning_rate = learning_rate), loss = loss, metrics = metrics)
# load the last saved weight from the training
SegFormer.load_weights(model_path)
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test2_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = SegFormer.evaluate( X_test2_roi, y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("SegFormer")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_SegFormer_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 856 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 44s 12s/step - IoU: 0.1763 - accuracy: 0.9448 - f1-score: 0.2998 - loss: 0.7301 - precision: 0.3810 - recall: 0.2556
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - IoU: 0.2317 - accuracy: 0.9559 - f1-score: 0.3763 - loss: 0.5610 - precision: 0.2720 - recall: 0.6102
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - IoU: 0.2570 - accuracy: 0.9698 - f1-score: 0.4090 - loss: 0.5098 - precision: 0.2885 - recall: 0.7018
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - IoU: 0.1292 - accuracy: 0.9826 - f1-score: 0.2288 - loss: 0.7489 - precision: 0.1862 - recall: 0.2967
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 24s 24s/step - IoU: 0.2697 - accuracy: 0.9229 - f1-score: 0.4248 - loss: 0.5810 - precision: 0.4403 - recall: 0.4104


## MODEL ChangeFormer1

In [76]:
from tensorflow import keras
from tensorflow.keras import layers, Model
import tensorflow as tf
!git clone https://github.com/IMvision12/SegFormer-tf
!cd SegFormer-tf
!cp SegFormer-tf/models/* copied_models/
import keras
from keras import ops
import math
from tensorflow.keras.activations import sigmoid

fatal: destination path 'SegFormer-tf' already exists and is not an empty directory.


In [77]:
class Attention(keras.layers.Layer):
    def __init__(
        self,
        dim,
        num_heads,
        sr_ratio,
        qkv_bias=False,
        attn_drop=0.0,
        proj_drop=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = self.dim // self.num_heads

        self.units = self.num_heads * self.head_dim
        self.sqrt_of_units = math.sqrt(self.head_dim)

        self.q = keras.layers.Dense(self.units)
        self.k = keras.layers.Dense(self.units)
        self.v = keras.layers.Dense(self.units)

        self.attn_drop = keras.layers.Dropout(attn_drop)

        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = keras.layers.Conv2D(
                filters=dim, kernel_size=sr_ratio, strides=sr_ratio, name='sr',
            )
            self.norm = keras.layers.LayerNormalization(epsilon=1e-05)

        self.proj = keras.layers.Dense(dim)
        self.proj_drop = keras.layers.Dropout(proj_drop)

    def call(self, x, H, W):
        get_shape = ops.shape(x)
        B = get_shape[0]
        C = get_shape[2]

        q = self.q(x)
        q = ops.reshape(q, (B, -1, self.num_heads, self.head_dim))
        q = ops.transpose(q, axes=[0, 2, 1, 3])  # Shape: [B, nh, T_q, hd]

        if self.sr_ratio > 1:
            x = ops.reshape(x, (B, H, W, C))
            x = self.sr(x)
            x = ops.reshape(x, (B, -1, C))
            x = self.norm(x)

        k = self.k(x)
        k = ops.reshape(k, (B, -1, self.num_heads, self.head_dim))
        k = ops.transpose(k, axes=[0, 2, 1, 3])  # Shape: [B, nh, T_k, hd]

        v = self.v(x)
        v = ops.reshape(v, (B, -1, self.num_heads, self.head_dim))
        v = ops.transpose(v, axes=[0, 2, 1, 3])  # Shape: [B, nh, T_v, hd]

        # Transpose k to [B, nh, hd, T_k] for correct matrix multiplication
        k_transposed = ops.transpose(k, axes=[0, 1, 3, 2])

        # Compute attention scores: [B, nh, T_q, T_k]
        attn = ops.matmul(q, k_transposed)
        scale = ops.cast(self.sqrt_of_units, dtype=attn.dtype)
        attn = attn / scale

        attn = ops.softmax(attn, axis=-1)
        attn = self.attn_drop(attn)

        # Multiply with value vectors
        x = ops.matmul(attn, v)  # Shape: [B, nh, T_q, hd]
        x = ops.transpose(x, axes=[0, 2, 1, 3])
        x = ops.reshape(x, (B, -1, self.units))
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [78]:
from copied_models.utils import DropPath

class DWConv(keras.layers.Layer):
    def __init__(self, filters=768, **kwargs):
        super().__init__(**kwargs)
        self.dwconv = keras.layers.Conv2D(
            filters=filters,
            kernel_size=3,
            strides=1,
            padding="same",
            groups=filters,
        )

    def call(self, x, H, W):
        get_shape_1 = ops.shape(x)
        x = ops.reshape(x, (get_shape_1[0], H, W, get_shape_1[-1]))
        x = self.dwconv(x)
        get_shape_2 = ops.shape(x)
        x = ops.reshape(
            x, (get_shape_2[0], get_shape_2[1] * get_shape_2[2], get_shape_2[3])
        )
        return x


class Mlp(keras.layers.Layer):
    def __init__(
        self,
        in_features,
        hidden_features=None,
        out_features=None,
        drop=0.0,
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = keras.layers.Dense(hidden_features)
        self.dwconv = DWConv(hidden_features)
        self.act = keras.layers.Activation("gelu")
        self.fc2 = keras.layers.Dense(out_features)
        self.drop = keras.layers.Dropout(drop)

    def call(self, x, H, W):
        x = self.fc1(x)
        x = self.dwconv(x, H=H, W=W)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class Block(keras.layers.Layer):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        qkv_bias=False,
        drop=0.0,
        attn_drop=0.0,
        drop_path=0.0,
        sr_ratio=1,
    ):
        super().__init__()
        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-05)
        self.attn = Attention(
            dim,
            num_heads,
            sr_ratio,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.drop_path = DropPath(drop_path)
        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-05)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            drop=drop,
        )

    def call(self, x, H, W):
        # Apply LayerNormalization and Attention layer
        attn_output_norm = self.norm1(x)
        attn_output = self.attn(attn_output_norm, H=H, W=W)
        attn_output_with_drop = self.drop_path(attn_output)
        x = x + attn_output_with_drop

        # Apply LayerNormalization and MLP layer
        mlp_output_norm = self.norm2(x)
        mlp_output = self.mlp(mlp_output_norm, H=H, W=W)
        mlp_output_with_drop = self.drop_path(mlp_output)
        x = x + mlp_output_with_drop

        return x



class OverlapPatchEmbed(keras.layers.Layer):
    def __init__(
        self, img_size=224, patch_size=7, stride=4, filters=768, **kwargs
    ):
        super().__init__(**kwargs)
        self.pad = keras.layers.ZeroPadding2D(padding=patch_size // 2)
        self.conv = keras.layers.Conv2D(
            filters=filters,
            kernel_size=patch_size,
            strides=stride,
            padding="VALID",
            name='proj',
        )
        self.norm = keras.layers.LayerNormalization(epsilon=1e-05)

    def call(self, x):
        x = self.conv(self.pad(x))
        get_shapes = ops.shape(x)
        H = get_shapes[1]
        W = get_shapes[2]
        C = get_shapes[3]
        x = ops.reshape(x, (-1, H * W, C))
        x = self.norm(x)
        return x, H, W

In [79]:
class MixVisionTransformer(keras.layers.Layer):
    def __init__(
        self,
        img_size=256,
        embed_dims=[64, 128, 256, 512, 1024],  # Updated embed_dims
        num_heads=[1, 2, 4, 8, 8],  # Updated num_heads
        mlp_ratios=[4, 4, 4, 4, 4],  # Updated mlp_ratios
        qkv_bias=False,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.0,
        depths=[3, 4, 6, 3, 3],  # Updated depths
        sr_ratios=[8, 8, 4, 2, 1],  # Updated sr_ratios
    ):
        super().__init__()
        self.depths = depths
        # patch_embed
        self.patch_embed1 = OverlapPatchEmbed(
            img_size=img_size,
            patch_size=7,
            stride=4,
            filters=embed_dims[0],
        )
        self.patch_embed2 = OverlapPatchEmbed(
            img_size=img_size // 4,
            patch_size=3,
            stride=2,
            filters=embed_dims[1],
        )
        self.patch_embed3 = OverlapPatchEmbed(
            img_size=img_size // 8,
            patch_size=3,
            stride=2,
            filters=embed_dims[2],
        )
        self.patch_embed4 = OverlapPatchEmbed(
            img_size=img_size // 16,
            patch_size=3,
            stride=2,
            filters=embed_dims[3],
        )
        self.patch_embed5 = OverlapPatchEmbed(  # New patch_embed5
            img_size=img_size // (4 * 2 * 2 * 2),  # Input size for stage 5
            patch_size=3,
            stride=2,
            filters=embed_dims[4],
        )
        dpr = [x for x in ops.linspace(0.0, drop_path_rate, sum(depths))]
        cur = 0
        self.block1 = [
            Block(
                dim=embed_dims[0],
                num_heads=num_heads[0],
                mlp_ratio=mlp_ratios[0],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[0],
            )
            for i in range(depths[0])
        ]
        self.norm1 = keras.layers.LayerNormalization(epsilon=1e-05)

        cur += depths[0]
        self.block2 = [
            Block(
                dim=embed_dims[1],
                num_heads=num_heads[1],
                mlp_ratio=mlp_ratios[1],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[1],
            )
            for i in range(depths[1])
        ]
        self.norm2 = keras.layers.LayerNormalization(epsilon=1e-05)

        cur += depths[1]
        self.block3 = [
            Block(
                dim=embed_dims[2],
                num_heads=num_heads[2],
                mlp_ratio=mlp_ratios[2],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[2],
            )
            for i in range(depths[2])
        ]
        self.norm3 = keras.layers.LayerNormalization(epsilon=1e-05)

        cur += depths[2]
        self.block4 = [
            Block(
                dim=embed_dims[3],
                num_heads=num_heads[3],
                mlp_ratio=mlp_ratios[3],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[3],
            )
            for i in range(depths[3])
        ]
        self.norm4 = keras.layers.LayerNormalization(epsilon=1e-05)

        cur += depths[3] # Adjust cur for block5
        self.block5 = [  # New block5
            Block(
                dim=embed_dims[4],
                num_heads=num_heads[4],
                mlp_ratio=mlp_ratios[4],
                qkv_bias=qkv_bias,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[cur + i],
                sr_ratio=sr_ratios[4],
            )
            for i in range(depths[4])
        ]
        self.norm5 = keras.layers.LayerNormalization(epsilon=1e-05)

    def call_features(self, x):
        B = ops.shape(x)[0]
        outs = []

        # stage 1
        x, H, W = self.patch_embed1(x)
        for i, blk in enumerate(self.block1):
            x = blk(x, H=H, W=W)
        x = self.norm1(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # stage 2
        x, H, W = self.patch_embed2(x)
        for i, blk in enumerate(self.block2):
            x = blk(x, H=H, W=W)
        x = self.norm2(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # stage 3
        x, H, W = self.patch_embed3(x)
        for i, blk in enumerate(self.block3):
            x = blk(x, H=H, W=W)
        x = self.norm3(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # stage 4
        x, H, W = self.patch_embed4(x)
        for i, blk in enumerate(self.block4):
            x = blk(x, H=H, W=W)
        x = self.norm4(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        # stage 5  # New stage 5
        x, H, W = self.patch_embed5(x)
        for i, blk in enumerate(self.block5):
            x = blk(x, H=H, W=W)
        x = self.norm5(x)
        x = ops.reshape(x, (B, H, W, ops.shape(x)[-1]))
        outs.append(x)

        return outs

    def call(self, x):
        x = self.call_features(x)
        return x

In [80]:
from copied_models.Head import SegFormerHead
from copied_models.utils import ResizeLayer

MODEL_CONFIGS = {
    "mit_b1": {
        "embed_dims": [64, 128, 320, 512, 1024],
        "depths": [2, 2, 2, 2, 2],
        "decode_dim": 512,
    }}

import tensorflow as tf

def add_tensors_in_lists(X1, X2):
  """Adds tensors in two lists element-wise.

  Args:
    X1: The first list of Keras tensors.
    X2: The second list of Keras tensors.

  Returns:
    A list containing the added tensors.
  """
  added_tensors = []
  for i in range(len(X1)):  # Assuming X1 and X2 have the same length
    added_tensors.append(tf.keras.layers.Add()([X1[i], X2[i]]))
  return added_tensors

In [81]:
# Size of the tiles/patches
size1 = 256 # This line takes the value of the 3rd index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 128.
size2 = 256

# Image bands
img_bands1 = 11 # This line takes the value of the 4th index which in this is taken from X_train shape = 1119, 128, 128, 11, that is 11.

In [82]:
def Changeformer(input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), num_classes=1):

    #Define the inputs
    input_1 = Input(shape=input_size1, name='input_1')  # Pre-event optical
    input_2 = Input(shape=input_size2, name='input_2')  # Post-event optical
    # 2. Create Encoder and Decoder
    encoder = MixVisionTransformer(
        img_size=input_size1[1],
        embed_dims=MODEL_CONFIGS["mit_b1"]["embed_dims"],
        depths=MODEL_CONFIGS["mit_b1"]["depths"])
    X_1 = encoder(input_1)
    X_2 = encoder(input_2)

    x = add_tensors_in_lists(X_1, X_2)

    # Assuming a corresponding decoder architecture
    x = SegFormerHead(
        num_classes=num_classes,
        decode_dim=MODEL_CONFIGS["mit_b1"]["decode_dim"],
    )(x)

    x = ResizeLayer(input_size1[0], input_size1[1])(x)
    x = sigmoid(x)

    return Model(inputs=[input_1, input_2], outputs=x)

In [83]:
# fix random seed for reproducibility
np.random.seed(42)
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
fiilter = 64
learning_rate = 10e-5

# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 4

dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []

model_path = f'/content/drive/MyDrive/landslide/Results/weights/Changeformer_size_{size1}_filters_{fiilter}_batch_size_{batch}_lr_{learning_rate}.keras'
# load unet to evaluate the test data
ChangeForm = Changeformer(input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), num_classes=1)
ChangeForm.compile(optimizer = Adam(learning_rate = learning_rate), loss = loss, metrics = metrics)
# load the last saved weight from the training
ChangeForm.load_weights(model_path)
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test2_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = ChangeForm.evaluate( [X_test1_roi, X_test2_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("ChangeForm")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_ChangeForm_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 512 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 35s 11s/step - IoU: 0.1185 - accuracy: 0.9286 - f1-score: 0.2116 - loss: 0.7838 - precision: 0.2204 - recall: 0.2410
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - IoU: 0.0232 - accuracy: 0.9055 - f1-score: 0.0453 - loss: 0.9421 - precision: 0.0291 - recall: 0.1029
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - IoU: 0.3253 - accuracy: 0.9794 - f1-score: 0.4909 - loss: 0.4541 - precision: 0.3888 - recall: 0.6660
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - IoU: 0.1248 - accuracy: 0.9892 - f1-score: 0.2219 - loss: 0.7945 - precision: 0.2949 - recall: 0.1779
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - IoU: 0.2925 - accuracy: 0.9204 - f1-score: 0.4527 - loss: 0.5403 - precision: 0.4327 - recall: 0.4745


## CMFNet

In [84]:
size1 = 256
img_bands1= 11
size2 = 256
img_bands2 = 6

In [85]:
class SpectralNorm(tf.keras.constraints.Constraint):
    def __init__(self, n_iter=5):
        self.n_iter = n_iter

    def call(self, input_weights):
        w = tf.reshape(input_weights, (-1, input_weights.shape[-1]))
        u = tf.random.normal((w.shape[0], 1))
        for _ in range(self.n_iter):
            v = tf.matmul(w, u, transpose_a=True)
            v /= tf.norm(v)

            u = tf.matmul(w, v)
            u /= tf.norm(u)

        spec_norm = tf.matmul(u, tf.matmul(w, v), transpose_a=True)
        return input_weights/spec_norm

In [86]:
class EuclideanDistanceLayer(Layer):
    def __init__(self, **kwargs):
        super(EuclideanDistanceLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # Assuming inputs is a list of two feature maps with the same shape
        feature_map1, feature_map2 = inputs

        # Calculate the pixel-wise Euclidean distance
        squared_diff = tf.square(feature_map1 - feature_map2)
        sum_squared_diff = tf.reduce_sum(squared_diff, axis=-1, keepdims=True)
        euclidean_distance = tf.sqrt(sum_squared_diff)

        return euclidean_distance

In [87]:
class SelfAttention(Layer):
    def __init__(self, ksize=4, stride=4, ratio=8, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.ksize = ksize
        self.stride = stride
        self.ratio = ratio

    def build(self, input_shape):
        n, h, w, c = input_shape
        self.n_feats = h * w
        # Reduced channel dimension
        reduced_c = c // self.ratio
        self.conv_theta = Conv2D(reduced_c, 1, padding='same')
        self.conv_phi = Conv2D(reduced_c, 1, padding='same')
        self.conv_g = Conv2D(c, 1, padding='same')
        self.conv_gc = Conv2D(c, 1, padding='same')

        # Gating weights
        self.gate_sa = self.add_weight(name="gate_sa", shape=(1,), initializer="zeros", trainable=True)
        self.gate_ca = self.add_weight(name="gate_ca", shape=(1,), initializer="zeros", trainable=True)

    def call(self, x):
        n, h, w, c = x.shape
        theta = self.conv_theta(x)
        theta = tf.reshape(theta, (-1, self.n_feats, theta.shape[-1]))

        phi = self.conv_phi(x)
        phi_s = tf.nn.max_pool2d(phi, self.ksize, self.stride, padding='VALID')
        # The error was in this line. We need to divide n_feats by (ksize * stride)
        # and multiply stride because of the maxpool
        phi_s = tf.reshape(phi_s, (-1, self.n_feats // (self.ksize * self.stride), phi_s.shape[-1]))

        attn = tf.matmul(theta, phi_s, transpose_b=True)
        attn = tf.nn.softmax(attn)

        g = self.conv_g(x)
        g = tf.nn.max_pool2d(g, self.ksize, self.stride, padding='VALID')
        # Same correction as in line 30
        g = tf.reshape(g, (-1, self.n_feats // (self.ksize * self.stride), g.shape[-1]))

        attn_g = tf.matmul(attn, g)
        attn_g = tf.reshape(attn_g, (-1, h, w, attn_g.shape[-1]))

        # Compute attention for CAM (Channel Attention Module)
        theta_c = tf.transpose(theta, perm=[0, 2, 1])
        phi = tf.reshape(phi, (-1, self.n_feats, phi.shape[-1]))

        attn_c = tf.matmul(theta_c, phi)
        attn_c = tf.nn.softmax(attn_c)

        g_c = self.conv_gc(x)
        g_c = tf.reshape(g_c, (-1, self.n_feats*self.ratio, c//self.ratio))

        attn_c = tf.matmul(g_c, attn_c)
        attn_c = tf.reshape(attn_c, (-1, h, w, c))
        #attn_c = self.conv_attn_c(attn_c)

        output = x + self.gate_sa * attn_g + self.gate_ca * attn_c

        return output
    def get_config(self):
        config = super().get_config()
        config.update({"ksize": self.ksize, "stride": self.stride, "ratio": self.ratio})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [88]:
def res_conv_block(x, kernelsize, filters, dropout, batchnorm=True):
    shortcut = x  # Identity connection
    c= x.shape[3]

    # Main path
    res = layers.Conv2D(filters, kernelsize, strides=1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    if batchnorm:
        res = layers.BatchNormalization()(res)
    res = layers.ReLU()(res)
    if dropout > 0:
        res = layers.Dropout(dropout)(res)

    res = layers.Conv2D(filters, kernelsize, strides=1, padding='same', use_bias=False, kernel_initializer='he_normal')(res)
    if batchnorm:
        res = layers.BatchNormalization()(res)
    if dropout > 0:
        res = layers.Dropout(dropout)(res)


    # Shortcut path (if needed)
    if c != filters:
        shortcut = layers.Conv2D(filters, kernel_size=1, strides=1, padding='same', use_bias=False, kernel_initializer='he_normal',)(x)

    # Residual connection
    output = layers.add([shortcut, res])

    return output

In [89]:
#convolutional block
def conv_block(x, kernelsize, filters,  batchnorm=False):
    conv = Conv2D(filters, (kernelsize, kernelsize),  padding="same")(x)
    if batchnorm is True:
        conv = BatchNormalization(axis=3)(conv)
    conv = Activation("relu")(conv)
    return conv

#convolutional block1
def conv_block1(x, kernelsize, filters, dropout,  batchnorm=False):
    conv = Conv2D(filters, (kernelsize, kernelsize), padding="same", kernel_initializer='he_normal')(x)
    if batchnorm is True:
        conv = BatchNormalization(axis=3)(conv)
    if dropout > 0:
        conv = Dropout(dropout, seed=42)(conv)
    conv = Conv2D(filters, (kernelsize, kernelsize), padding="same", kernel_initializer='he_normal')(conv)
    if batchnorm is True:
        conv = BatchNormalization(axis=3)(conv)
    conv = Activation("relu")(conv)
    return conv

In [90]:
def inverted_residual_block(inputs, expansion_factor, output_channels, stride):
    input_channels = inputs.shape[-1]
    x = inputs

    # Expand phase
    if expansion_factor != 1:
        x = layers.Conv2D(input_channels * expansion_factor, 1, padding='same', use_bias=True, activation=keras.activations.hard_silu)(x)

    # Depthwise Convolution
    x = layers.DepthwiseConv2D(3, strides=stride,  padding='same', use_bias=True, activation=keras.activations.hard_silu)(x)

    # Linear bottleneck
    x = layers.Conv2D(output_channels, 1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if stride == 1 and input_channels == output_channels:
        x = layers.Add()([inputs, x])

    return x

In [91]:
def SFIM1(F1_pre, F2_post, name):
    mi = EuclideanDistanceLayer()([F1_pre, F2_post])
    # Apply sigmoid activation to get attention map ai
    ai = Activation('sigmoid')(mi)
    x = tf.keras.layers.Concatenate(axis=-1)([F1_pre, F2_post])
    conv = Conv2D(filters= F1_pre.shape[-1], kernel_size=(3, 3),  padding='same', kernel_initializer='he_normal')(x)
    # Elementwise multiplication of Gi and ai
    output = tf.keras.layers.Multiply()([conv, ai])
    return output

In [92]:
def channel_attention(input_feature, ratio=8):
    channel = input_feature.shape[-1]

    shared_layer_one = Dense(channel//ratio,
                             activation='relu',
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
    shared_layer_two = Dense(channel,
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')

    avg_pool = GlobalAveragePooling2D()(input_feature)
    avg_pool = Reshape((1,1,channel))(avg_pool)
    # Changed _keras_shape to shape
    assert avg_pool.shape[1:] == (1,1,channel)
    avg_pool = shared_layer_one(avg_pool)
    # Changed _keras_shape to shape
    assert avg_pool.shape[1:] == (1,1,channel//ratio)
    avg_pool = shared_layer_two(avg_pool)
    # Changed _keras_shape to shape
    assert avg_pool.shape[1:] == (1,1,channel)

    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1,1,channel))(max_pool)
    # Changed _keras_shape to shape
    assert max_pool.shape[1:] == (1,1,channel)
    max_pool = shared_layer_one(max_pool)
    # Changed _keras_shape to shape
    assert max_pool.shape[1:] == (1,1,channel//ratio)
    max_pool = shared_layer_two(max_pool)
    # Changed _keras_shape to shape
    assert max_pool.shape[1:] == (1,1,channel)

    cbam_feature = Add()([avg_pool,max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)

    return multiply([input_feature, cbam_feature])

In [93]:
def SFIM2(input_tensor, filters, name, dilation_rates=[6, 12, 18]):

    # 1x1 Convolution (Without dilation)
    conv_1x1 = Conv2D(filters, (1, 1), padding='same', use_bias=False, kernel_initializer=keras.initializers.HeNormal())(input_tensor)
    conv_1x1 = BatchNormalization()(conv_1x1)
    conv_1x1 = Activation('relu')(conv_1x1)

    # Atrous Convolutions with different dilation rates
    atrous_convs = []
    for rate in dilation_rates:
        x = Conv2D(filters, (3, 3), padding='same', dilation_rate=rate, use_bias=False, kernel_initializer=keras.initializers.HeNormal())(input_tensor)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        atrous_convs.append(x)

    # Global Average Pooling followed by 1x1 Convolution
    global_avg = GlobalAveragePooling2D()(input_tensor)
    global_avg = layers.Reshape((1, 1, global_avg.shape[-1]))(global_avg)
    global_avg = Conv2D(filters, (1, 1), padding='same', use_bias=True)(global_avg)
    global_avg = BatchNormalization()(global_avg)
    global_avg = Activation('relu')(global_avg)
    global_avg = UpSampling2D(size=(input_tensor.shape[1], input_tensor.shape[2]), interpolation="bilinear")(global_avg)

    # Concatenate all outputs
    x = Concatenate(axis=-1)([conv_1x1] + atrous_convs + [global_avg])
    x = channel_attention(x)

    # Final 1x1 convolution
    output = tf.keras.layers.Conv2D(filters, (1, 1), padding='same', use_bias=False, kernel_initializer=keras.initializers.HeNormal(), activation='relu',  name=name)(x)
    return output

In [94]:
def UpSample(x, filters, interpolation='bilinear'):
    x = layers.UpSampling2D(size=2, interpolation=interpolation)(x)
    x = layers.Conv2D(
        filters, kernel_size=3, padding="same", use_bias=False, kernel_initializer='he_normal'
    )(x)
    return x

In [95]:
#Version 1 of FFUM1
def FFUM1(X_i, X_i_plus_1, filters , name):

    # Step 1: Apply two deconvolutions with 1x1 convolution and batch normalization
    conv1 = UpSample(X_i_plus_1, filters)

    # Step 2: Concatenate the feature maps
    S1 = Concatenate(axis=-1)([X_i, conv1])
    # Convolutional layers for g and x
    W_g = Conv2D(filters, kernel_size=1, strides=1, padding='same', use_bias=True, kernel_initializer='he_normal')(X_i)
    W_g = BatchNormalization()(W_g)

    W_x = Conv2D(filters, kernel_size=1, strides=1, padding='same', use_bias=True, kernel_initializer='he_normal')(conv1)
    W_x = BatchNormalization()(W_x)

    # Attention map generation
    psi = relu((Add()([W_g, W_x])))  # Add and apply ReLU
    psi = Conv2D(1, kernel_size=1, strides=1, padding='same', use_bias=True, kernel_initializer='he_normal')(psi)
    psi = BatchNormalization()(psi)
    psi = tf.keras.layers.Activation('sigmoid')(psi)  # Apply sigmoid

    # Apply attention and return
    attended_x = Multiply()([S1, psi])

    # Apply a 3x3 convolution
    output = res_conv_block(attended_x, kernelsize=3, filters=filters, dropout=0.3, batchnorm=True)

    return output

In [96]:
#Main branch encoder
def Main_branch_CSAencoder_1(filtersFirstLayer, input_shape, batchnorm=True):

    # Define input within the function
    input_tensor = Input(shape=input_shape)
    # Initial stem Convolution Layer
    F1 = layers.Conv2D(filtersFirstLayer, 3, strides=(1, 1), padding='same', use_bias=False)(input_tensor)
    F1 = layers.BatchNormalization()(F1)
    F1 = Dropout(0.1)(F1)
    F1 = layers.Conv2D(filtersFirstLayer, 3, strides=(1, 1), padding='same', use_bias=False)(F1)
    F1 = layers.BatchNormalization()(F1)
    F1 = layers.ReLU()(F1)

    # Second layer encoder

    F2 = inverted_residual_block(F1, expansion_factor=4, output_channels=filtersFirstLayer, stride=1)
    F2 = MaxPooling2D(pool_size=(2, 2), name='F2_layer')(F2)
    F2_att = SelfAttention(ksize=4, stride=4, ratio=4) (F2)
    #print(F2.shape)

    # Third layer encoder

    F3 = inverted_residual_block(F2, expansion_factor=4, output_channels=filtersFirstLayer*2, stride=1)
    for _ in range(1):
        F3 = inverted_residual_block(F3, expansion_factor=4, output_channels=filtersFirstLayer*2, stride=1)
    F3 = MaxPooling2D(pool_size=(2, 2), name='F3_layer')(F3)
    F3_att = SelfAttention(ksize=2, stride=2, ratio=4) (F3)
    #F3 = SALayer(filtersFirstLayer*2)(F3)
    #print(F3.shape)

    # Fourth layer encoder

    F4 = inverted_residual_block(F3, expansion_factor=4, output_channels=filtersFirstLayer*4, stride=1)
    for _ in range(1):
        F4 = inverted_residual_block(F4, expansion_factor=4, output_channels=filtersFirstLayer*4, stride=1)
    F4 = MaxPooling2D(pool_size=(2, 2), name='F4_layer')(F4)
    F4_att = SelfAttention(ksize=1, stride=1, ratio=4) (F4)
    #F4 = SALayer(filtersFirstLayer*4)(F4)
    #print(F4.shape)

    # Fifth layer encoder

    F5 = inverted_residual_block(F4, expansion_factor=4, output_channels=filtersFirstLayer*8, stride=1)
    for _ in range(2):
        F5 = inverted_residual_block(F5, expansion_factor=4, output_channels=filtersFirstLayer*8, stride=1)
    F5 = MaxPooling2D(pool_size=(2, 2), name='F5_layer')(F5)
    F5_att = SelfAttention(ksize=1, stride=1, ratio=4) (F5)
    #F5 = SALayer(filtersFirstLayer*8)(F5)
    #print(F5.shape)

    F6 = inverted_residual_block(F5, expansion_factor=4, output_channels=filtersFirstLayer*16, stride=1)
    for _ in range(2):
        F6 = inverted_residual_block(F6, expansion_factor=4, output_channels=filtersFirstLayer*16, stride=1)
    F6 = MaxPooling2D(pool_size=(2, 2), name='F6_layer')(F6)

    model = Model(inputs= input_tensor, outputs=[F1, F2_att, F3_att, F4_att, F5_att, F6])
    return model

In [97]:
def Siamese_CSA_1(filtersFirstLayer, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2),batchnorm=True):

    #Define the inputs
    input_1 = Input(shape=input_size1, name='input_1')  # Pre-event optical
    input_2 = Input(shape=input_size2, name='input_2')  # Post-event optical
    input_3 = Input(shape=input_size3, name='input_3')  # Topographic

    # Concatenate optical images with topographic data
    input_cat1 = Concatenate(axis=-1)([input_1, input_3])  # Pre-event + Topo
    input_cat2 = Concatenate(axis=-1)([input_2, input_3])  # Post-event + Topo

    # Define shared encoder with correct input shape
    encoder_model = Main_branch_CSAencoder_1(filtersFirstLayer, input_shape = input_cat1.shape[1:])

    # Apply encoder to both concatenated inputs
    Features1 = encoder_model(input_cat1)
    Features2 = encoder_model(input_cat2)

    # Unpacking encoder outputs
    F1_1, F2_1, F3_1, F4_1, F5_1, F6_1 = Features1
    F1_2, F2_2, F3_2, F4_2, F5_2, F6_2 = Features2

    #1st layer of the second branch

    sfi1_0 = SFIM1(F1_1, F1_2, name='SFI1_0')
    #print(sfi1_0.shape)

    # Second layer encoder

    sfi1_1 = SFIM1(F2_1, F2_2, name='SFI1_1')
    #print(sfi1_1.shape)

    # Third layer encoder

    sfi1_2 = SFIM1(F3_1, F3_2, name='SFI1_2')
    #print(sfi1_2.shape)

    # Fourth layer encoder

    sfi1_3 = SFIM1(F4_1, F4_2, name='SFI1_3')
    #print(sfi1_3.shape)

    # Fifth layer encoder
    sfi1_4  = SFIM1(F5_1, F5_2, name='SFI1_4')
    #print(sfi1_4.shape)

    # bottlneck layer encoder
    #sfi1_5  = layers.Add(name='fuse')([F6_1, F6_2])
    sfi1_5 = Concatenate(axis=-1)([F6_1, F6_2])
    #print(sfi1_5.shape)

    #Shared Feature information module 2
    sfi2 = SFIM2(sfi1_5, filters=filtersFirstLayer*16, name='SFI2')
    #print(sfi2.shape)

    #1st layer Decoder
    Y_2 = FFUM1(sfi1_4, sfi2, filters=filtersFirstLayer*8, name='FFUM1_1')
    #print(Y_2.shape)

    #2nd layer Decoder
    Y_3 = FFUM1(sfi1_3, Y_2, filters=filtersFirstLayer*4, name='FFUM1_2')
    #print(Y_3.shape)

    #3rd layer Decoder
    Y_4 = FFUM1(sfi1_2, Y_3, filters=filtersFirstLayer*2, name='FFUM1_3')

    #4th layer Decoder
    Y_5 = FFUM1(sfi1_1, Y_4, filters=filtersFirstLayer, name='FFUM1_4')

    #Classification Layer
    Y_6 = UpSample(Y_5, filters=filtersFirstLayer)
    merge = Concatenate(axis=-1)([Y_6, sfi1_0])
    Y_6 = conv_block1(merge, kernelsize=3, filters=filtersFirstLayer, dropout=0.2, batchnorm=True)

    #print(Y_4.shape)
    Final = conv_block(Y_6, kernelsize=1, filters=1, batchnorm=False)
    Final = Activation('sigmoid')(Final)
    #print(Final.shape)

    return Model(inputs=[input_1, input_2, input_3], outputs=Final)

In [98]:
# Here we define the evaluation metrics - Precision, Recall, FScore, IoU
metrics = [sm.metrics.Precision(threshold=0.5),sm.metrics.Recall(threshold=0.5),sm.metrics.FScore(threshold=0.5,beta=1, name= 'f1-score'),sm.metrics.IOUScore(threshold=0.5, name= 'IoU'), 'accuracy']
loss= loss
# Number of filters. We set a range of the number of filters for the convolutional layers.
fiilter = 64
learning_rate = 10e-5

# Batch sizes. This considers how many patches the model will take during the training phases. A value of 4 means 4 patches of images (from 1119) will be taken as a batch while training simultaneously.
batch = 4

dic["model"] = [] # Name of the model
dic["ROI"] = [] # ROI label
# Metrics on the test set. It will save the metrics after evaluating the model on the test set.
dic["precision_area"] = []
dic["recall_area"] = []
dic["f1_score_area"] = []
dic["miou_area"] = []
dic["oa_area"] = []

# load unet to evaluate the test data
CMFNet = Siamese_CSA_1(filtersFirstLayer=fiilter, input_size1 = (size1,size1,img_bands1), input_size2 = (size1,size1,img_bands1), input_size3 = (size2,size2,img_bands2),batchnorm=True)
CMFNet.compile(optimizer = optimizer, loss = loss, metrics = metrics)
# load the last saved weight from the training
CMFNet.load_weights('/content/drive/MyDrive/landslide/Selection_model/Model_1_V1 (With FFUM V1)/Siamese_CSA_size_256_filters_64_batch_size_4_lr_0.0001.weights.h5')
# Get unique ROI values
unique_rois = np.unique(labels)

# Iterate through each ROI and evaluate
for roi in unique_rois:
    # Get indices of images belonging to the current ROI
    roi_indices = np.where(labels == roi)[0]

    # Extract data for the current ROI
    X_test1_roi = X_test1[roi_indices]
    X_test2_roi = X_test2[roi_indices]
    X_test3_roi = X_test3[roi_indices]
    y_test_roi = y_test[roi_indices]
    print(f"Processing ROI: {roi}")
    print(f"X_test1_roi shape: {X_test2_roi.shape}")

    # Evaluate the model on the current ROI
    att_1 = CMFNet.evaluate( [X_test1_roi, X_test2_roi, X_test3_roi], y_test_roi)

    # save results on the dictionary and then output them all in a Excel CSV file.
    dic["model"].append("CMFNet")
    dic["ROI"].append(roi)  # Store the ROI label
    dic["precision_area"].append(att_1[1])
    dic["recall_area"].append(att_1[2])
    dic["f1_score_area"].append(att_1[3])
    dic["miou_area"].append(att_1[4])
    dic["oa_area"].append(att_1[5])

# Convert results to a dataframe
results = pd.DataFrame(dic)
# Export as csv
results.to_csv(f'/content/drive/MyDrive/landslide/Results/csv/Test_results_CMFNet_by_ROI.csv', index = False)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 491 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Processing ROI: 0
X_test1_roi shape: (39, 256, 256, 11)
2/2 ━━━━━━━━━━━━━━━━━━━━ 129s 35s/step - IoU: 0.2913 - accuracy: 0.9543 - f1-score: 0.4510 - loss: 0.8213 - precision: 0.5388 - recall: 0.3891
Processing ROI: 1
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - IoU: 0.2546 - accuracy: 0.9472 - f1-score: 0.4058 - loss: 0.8870 - precision: 0.2690 - recall: 0.8265
Processing ROI: 2
X_test1_roi shape: (1, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - IoU: 0.3751 - accuracy: 0.9857 - f1-score: 0.5455 - loss: 0.9284 - precision: 0.5175 - recall: 0.5768
Processing ROI: 3
X_test1_roi shape: (3, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 27s 27s/step - IoU: 0.2618 - accuracy: 0.9866 - f1-score: 0.4150 - loss: 0.9578 - precision: 0.3345 - recall: 0.5465
Processing ROI: 4
X_test1_roi shape: (27, 256, 256, 11)
1/1 ━━━━━━━━━━━━━━━━━━━━ 86s 86s/step - IoU: 0.4089 - accuracy: 0.9456 - f1-score: 0.5804 - loss: 0.7450 - precision: 0.6236 - recall: 0.5428


## Vizualize Prediction on test data

In [ ]:
def adaptive_threshold_selection(DI, GI):
    adaptive_thresholds = [] # Initialize sum of image-specific thresholds
    N, X, Y = DI.shape[0], DI.shape[1], DI.shape[2]
    for i in range(N):
        # Compute tumor image TI_i by multiplying training image with ground truth image
        TI_i = DI[i] * GI[i]  # Perform element-wise multiplication

        # Calculate the sum of pixel intensities in the tumor image
        pixel_sum = 0
        non_zero_count = 0

        # Calculate sum of pixel intensities and count non-zero pixels in TI_i
        for x in range(X):  # assuming X and Y are predefined
            for y in range(Y):
                pixel_sum += TI_i[x][y]
                if np.all(TI_i[x][y] != 0):
                    non_zero_count += 1

        # Calculate the image-specific threshold ist_i
        if non_zero_count != 0:
            ist_i = pixel_sum / non_zero_count
        else:
            ist_i = 0  # handle case to avoid division by zero

        # Accumulate ist_i to sum_ist
        adaptive_thresholds.append(ist_i)
    # Compute adaptive threshold ath by averaging sum of image-specific thresholds

    ath = np.mean(adaptive_thresholds)
    print(ath)

    return ath

In [ ]:
# Call the function to get the adaptive threshold
#adaptive_threshold = adaptive_threshold_selection(X_train2, y_train)
#DI_combined1 = np.concatenate([X_train1, X_train2], axis=-1)
#DI_combined2 = np.concatenate([X_train1, X_train2, X_train3], axis=-1)
DI_combined3 = np.concatenate([X_train2, X_train3], axis=-1)
#adaptive_threshold1 = adaptive_threshold_selection(DI_combined1, y_train)
#adaptive_threshold2 = adaptive_threshold_selection(DI_combined2, y_train)
adaptive_threshold3 = adaptive_threshold_selection(DI_combined3, y_train)
#print("Adaptive Threshold:", adaptive_threshold)
#print("Adaptive Threshold:", adaptive_threshold1)
#print("Adaptive Threshold:", adaptive_threshold2)
print("Adaptive Threshold:", adaptive_threshold3)

0.4183133707149583
Adaptive Threshold: 0.4183133707149583


In [ ]:
# Call the function to get the adaptive threshold
adaptive_threshold = adaptive_threshold_selection(X_train2, y_train)
DI_combined1 = np.concatenate([X_train1, X_train2], axis=-1)
DI_combined2 = np.concatenate([X_train1, X_train2, X_train3], axis=-1)
DI_combined3 = np.concatenate([X_train2, X_train3], axis=-1)
adaptive_threshold1 = adaptive_threshold_selection(DI_combined1, y_train)
adaptive_threshold2 = adaptive_threshold_selection(DI_combined2, y_train)
adaptive_threshold3 = adaptive_threshold_selection(DI_combined3, y_train)
print("Adaptive Threshold:", adaptive_threshold)
print("Adaptive Threshold:", adaptive_threshold1)
print("Adaptive Threshold:", adaptive_threshold2)
print("Adaptive Threshold:", adaptive_threshold3)

0.4069052800251004
0.38675986882874414
0.39825385255497814
Adaptive Threshold: 0.4069052800251004
Adaptive Threshold: 0.38675986882874414
Adaptive Threshold: 0.39825385255497814


In [ ]:
no = 10
# Plot predictions on test set
fig, axarr = plt.subplots(10,12,figsize=(40,40))

for i in range(no):
    preds_train_0_0 = Transunet.predict(X_test2, verbose=0)
    preds_train_1_0 = Transunet.predict(np.fliplr(X_test2), verbose=0)
    preds_train_1_0 = np.fliplr(preds_train_1_0)
    preds_train_2_0 = Transunet.predict(np.flipud(X_test2), verbose=0)
    preds_train_2_0 = np.flipud(preds_train_2_0)
    preds_train_3_0 = Transunet.predict(np.fliplr(np.flipud(X_test2)), verbose=0)
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.4069052800251004).astype(np.uint8)

    preds_train_0_0 = SegFormer.predict(X_test2, verbose=0)
    preds_train_1_0 = SegFormer.predict(np.fliplr(X_test2), verbose=0)
    preds_train_1_0 = np.fliplr(preds_train_1_0)
    preds_train_2_0 = SegFormer.predict(np.flipud(X_test2), verbose=0)
    preds_train_2_0 = np.flipud(preds_train_2_0)
    preds_train_3_0 = SegFormer.predict(np.fliplr(np.flipud(X_test2)), verbose=0)
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t1_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.4069052800251004).astype(np.uint8)

    # Predictions with TTA (Test-Time Augmentation)
    preds_train_0_0 = ChangeForm.predict([X_test1, X_test2], verbose=0)  # Original orientation
    preds_train_1_0 = ChangeForm.predict([np.fliplr(X_test1), np.fliplr(X_test2)], verbose=0)  # Horizontal flip
    preds_train_1_0 = np.fliplr(preds_train_1_0)  # Flip back
    preds_train_2_0 = ChangeForm.predict([np.flipud(X_test1), np.flipud(X_test2)], verbose=0)  # Vertical flip
    preds_train_2_0 = np.flipud(preds_train_2_0)  # Flip back
    preds_train_3_0 = ChangeForm.predict([np.fliplr(np.flipud(X_test1)), np.fliplr(np.flipud(X_test2))], verbose=0)  # Both flips
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))  # Flip back
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t2_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.38675986882874414).astype(np.uint8)

    # Predictions with TTA (Test-Time Augmentation)
    preds_train_0_0 = SiAUnet.predict([X_test1, X_test2], verbose=0)  # Original orientation
    preds_train_1_0 = SiAUnet.predict([np.fliplr(X_test1), np.fliplr(X_test2)], verbose=0)  # Horizontal flip
    preds_train_1_0 = np.fliplr(preds_train_1_0)  # Flip back
    preds_train_2_0 = SiAUnet.predict([np.flipud(X_test1), np.flipud(X_test2)], verbose=0)  # Vertical flip
    preds_train_2_0 = np.flipud(preds_train_2_0)  # Flip back
    preds_train_3_0 = SiAUnet.predict([np.fliplr(np.flipud(X_test1)), np.fliplr(np.flipud(X_test2))], verbose=0)  # Both flips
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))  # Flip back
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t3_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.38675986882874414).astype(np.uint8)

    # Predictions with TTA (Test-Time Augmentation)
    preds_train_0_0 = Sia_SwinUnet.predict([X_test1, X_test2], verbose=0)  # Original orientation
    preds_train_1_0 = Sia_SwinUnet.predict([np.fliplr(X_test1), np.fliplr(X_test2)], verbose=0)  # Horizontal flip
    preds_train_1_0 = np.fliplr(preds_train_1_0)  # Flip back
    preds_train_2_0 = Sia_SwinUnet.predict([np.flipud(X_test1), np.flipud(X_test2)], verbose=0)  # Vertical flip
    preds_train_2_0 = np.flipud(preds_train_2_0)  # Flip back
    preds_train_3_0 = Sia_SwinUnet.predict([np.fliplr(np.flipud(X_test1)), np.fliplr(np.flipud(X_test2))], verbose=0)  # Both flips
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))  # Flip back
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t4_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.38675986882874414).astype(np.uint8)

    # Predictions with TTA (Test-Time Augmentation)
    preds_train_0_0 = CMFNet.predict([X_test1, X_test2, X_test3], verbose=0)  # Original orientation
    preds_train_1_0 = CMFNet.predict([np.fliplr(X_test1), np.fliplr(X_test2), np.fliplr(X_test3)], verbose=0)  # Horizontal flip
    preds_train_1_0 = np.fliplr(preds_train_1_0)  # Flip back
    preds_train_2_0 = CMFNet.predict([np.flipud(X_test1), np.flipud(X_test2), np.flipud(X_test3)], verbose=0)  # Vertical flip
    preds_train_2_0 = np.flipud(preds_train_2_0)  # Flip back
    preds_train_3_0 = CMFNet.predict([np.fliplr(np.flipud(X_test1)), np.fliplr(np.flipud(X_test2)), np.fliplr(np.flipud(X_test3))], verbose=0)  # Both flips
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))  # Flip back
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t5_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.39825385255497814).astype(np.uint8)

    # Predictions with TTA (Test-Time Augmentation)
    preds_train_0_0 = SDCUnet.predict(np.concatenate([X_test2, X_test3], axis=-1), verbose=0)
    preds_train_1_0 = SDCUnet.predict(np.fliplr(np.concatenate([X_test2, X_test3], axis=-1)), verbose=0)
    preds_train_1_0 = np.fliplr(preds_train_1_0)
    preds_train_2_0 = SDCUnet.predict(np.flipud(np.concatenate([X_test2, X_test3], axis=-1)), verbose=0)
    preds_train_2_0 = np.flipud(preds_train_2_0)
    preds_train_3_0 = SDCUnet.predict(np.fliplr(np.flipud(np.concatenate([X_test2, X_test3], axis=-1))), verbose=0)
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t6_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.4183133707149583).astype(np.uint8)

    # Predictions with TTA (Test-Time Augmentation)
    preds_train_0_0 = BBUNet.predict([X_test1, X_test2, X_test3], verbose=0)  # Original orientation
    preds_train_1_0 = BBUNet.predict([np.fliplr(X_test1), np.fliplr(X_test2), np.fliplr(X_test3)], verbose=0)  # Horizontal flip
    preds_train_1_0 = np.fliplr(preds_train_1_0)  # Flip back
    preds_train_2_0 = BBUNet.predict([np.flipud(X_test1), np.flipud(X_test2), np.flipud(X_test3)], verbose=0)  # Vertical flip
    preds_train_2_0 = np.flipud(preds_train_2_0)  # Flip back
    preds_train_3_0 = BBUNet.predict([np.fliplr(np.flipud(X_test1)), np.fliplr(np.flipud(X_test2)), np.fliplr(np.flipud(X_test3))], verbose=0)  # Both flips
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))  # Flip back
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t7_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.39825385255497814).astype(np.uint8)

    # Predictions with TTA (Test-Time Augmentation)
    preds_train_0_0 = GASA.predict([X_test1, X_test2, X_test3], verbose=0)  # Original orientation
    preds_train_1_0 = GASA.predict([np.fliplr(X_test1), np.fliplr(X_test2), np.fliplr(X_test3)], verbose=0)  # Horizontal flip
    preds_train_1_0 = np.fliplr(preds_train_1_0)  # Flip back
    preds_train_2_0 = GASA.predict([np.flipud(X_test1), np.flipud(X_test2), np.flipud(X_test3)], verbose=0)  # Vertical flip
    preds_train_2_0 = np.flipud(preds_train_2_0)  # Flip back
    preds_train_3_0 = GASA.predict([np.fliplr(np.flipud(X_test1)), np.fliplr(np.flipud(X_test2)), np.fliplr(np.flipud(X_test3))], verbose=0)  # Both flips
    preds_train_3_0 = np.fliplr(np.flipud(preds_train_3_0))  # Flip back
    # It's possible to change the 0.5 threshold to improve the results;
    preds_train_t8_1 = (((preds_train_0_0 + preds_train_1_0 + preds_train_2_0 + preds_train_3_0)/4) > 0.39825385255497814).astype(np.uint8)

    axarr[i, 0].imshow(X_test1[i][:,:,:3])
    axarr[i, 1].imshow(X_test2[i][:,:,:3])

    show(preds_train_t_1[i], ax=axarr[i, 2], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t1_1[i], ax=axarr[i, 3], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t2_1[i], ax=axarr[i, 4], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t3_1[i], ax=axarr[i, 5], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t4_1[i], ax=axarr[i, 6], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t5_1[i], ax=axarr[i, 7], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t6_1[i], ax=axarr[i, 8], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t7_1[i], ax=axarr[i, 9], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})
    show(preds_train_t8_1[i], ax=axarr[i, 10], contour=True, linestyles = 'solid', colors ='blue', linewidths=0.25, contour_label_kws = {"inline": False, "fontsize": 0})

    axarr[i, 11].imshow(np.squeeze(y_test[i]), cmap ='binary_r')

    if i == 0:
      axarr[i, 0].set_title("Pre image")
      axarr[i, 0].title.set_fontsize(22)
      axarr[i, 1].set_title("Post image")
      axarr[i, 2].title.set_fontsize(22)
      axarr[i, 2].set_title("Transunet")
      axarr[i, 3].title.set_fontsize(22)
      axarr[i, 3].set_title("SegFormer")
      axarr[i, 4].title.set_fontsize(22)
      axarr[i, 4].set_title("Changeformer")
      axarr[i, 5].title.set_fontsize(22)
      axarr[i, 5].set_title("SiAUnet")
      axarr[i, 6].title.set_fontsize(22)
      axarr[i, 6].set_title("Sia-SwinUnet")
      axarr[i, 7].title.set_fontsize(22)
      axarr[i, 7].set_title("CMFNet")
      axarr[i, 8].title.set_fontsize(22)
      axarr[i, 8].set_title("SDCUnet")
      axarr[i, 9].title.set_fontsize(22)
      axarr[i, 9].set_title("BBUNet")
      axarr[i, 10].title.set_fontsize(22)
      axarr[i, 10].set_title("Ours")
      axarr[i, 11].title.set_fontsize(22)
      axarr[i, 11].set_title("Label")

    axarr[i, 0].set(xticks=[], yticks=[])
    axarr[i, 1].set(xticks=[], yticks=[])
    axarr[i, 2].set(xticks=[], yticks=[])
    axarr[i, 3].set(xticks=[], yticks=[])
    axarr[i, 4].set(xticks=[], yticks=[])
    axarr[i, 5].set(xticks=[], yticks=[])
    axarr[i, 6].set(xticks=[], yticks=[])
    axarr[i, 7].set(xticks=[], yticks=[])
    axarr[i, 8].set(xticks=[], yticks=[])
    axarr[i, 9].set(xticks=[], yticks=[])
    axarr[i, 10].set(xticks=[], yticks=[])
    axarr[i, 11].set(xticks=[], yticks=[])

# Adjusting subplot layout for better positioning
plt.tight_layout()
plt.subplots_adjust(bottom=0.108)
plt.savefig("/content/drive/MyDrive/landslide/Results/plots/Predictions.png")
plt.show()